In [105]:
import pandas as pd
import datetime
import xlsxwriter
import os
import seaborn as sns
import numpy as np
import sys
from tjn_tools.data_processing import *
import tjn_tools
from openpyxl import *
%matplotlib
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import gmean
import squarify

from config_new import *

Using matplotlib backend: TkAgg


In [106]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [107]:
##Import data - this file comes from taking the combined_output file, running it through Mirek's pipeline to add more variables, and then exporting as dta file
file_name= "../../data/final/analysis/2018combined_output_additionalcountries.xlsx"
file_name_old = "../../data/final/analysis/2018combined_output.xlsx"
# pd.read_stata("../../data/final/analysis/230807 all data sotj2023.dta")
sotj_2023 = pd.read_excel(file_name_old) # change to file_name_old to see old estimates

sotj_2023.dropna(subset=["iso3"], inplace=True)
sotj_2023["OW: Tax revenue loss (% total)"] = sotj_2023["OW: Tax revenue loss (USD million)"]/(sotj_2023["OW: Tax revenue loss (USD million)"].sum())
sotj_2023["Loss TA using CIT (% total)"] = sotj_2023["Loss TA using CIT (USD million)"]/(sotj_2023["Loss TA using CIT (USD million)"].sum())
sotj_2023["Loss Total using CIT (% total)"] = sotj_2023["Loss Total using CIT (USD million)"]/(sotj_2023["Loss Total using CIT (USD million)"].sum())
sotj_2023["Harm TA using CIT (% total)"] = sotj_2023["Harm TA: Total using CIT (USD million)"]/(sotj_2023["Harm TA: Total using CIT (USD million)"].sum())
sotj_2023["GDP share"] = sotj_2023["GDP"]/(sotj_2023["GDP"].sum())
sotj_2023["Population share"] = sotj_2023["population"]/(sotj_2023["population"].sum())
sotj_2023.loc[sotj_2023["Profit loss (M) - dom"] > 0, "Profit shifted out"]  = sotj_2023["Profit loss (M) - for lose"] + sotj_2023["Profit loss (M) - dom"]
sotj_2023.loc[sotj_2023["Profit loss (M) - dom"] <= 0, "Profit shifted out"]  = sotj_2023["Profit loss (M) - for lose"]

sotj_2023.loc[sotj_2023["Profit loss (M) - dom"] < 0, "Profit shifted in"]  = (sotj_2023["Profit loss (M) - for gain"] + sotj_2023["Profit loss (M) - dom"]) * (-1)
sotj_2023.loc[sotj_2023["Profit loss (M) - dom"] >= 0, "Profit shifted in"]  = sotj_2023["Profit loss (M) - for gain"] * (-1)

#sotj_2023.dropna(subset=["Loss Total using CIT (USD million)", "Profit shifted in"], how="all", inplace=True)

In [108]:
iso3_with_nan_income = sotj_2023.loc[sotj_2023["income_class"].isna(), "iso3"]
sorted_iso3_with_nan_income = iso3_with_nan_income.sort_values()
print(sorted_iso3_with_nan_income)

4      AIA
225    ALA
226    ANT
227    ATF
228    BES
229    BLM
230    BVT
231    CCK
223    COK
232    CXR
233    ESH
234    FLK
190    GGY
220    GLP
219    GUF
235    HMD
236    IOT
192    JEY
17     MSR
237    MTQ
238    MYT
239    NFK
240    NIU
241    PAL
242    PCN
243    REU
244    SGS
245    SHN
246    SJM
247    SPM
248    TKL
249    TMP
200    TWN
250    UMI
251    VAT
224    WLF
252    WSH
253    XXK
Name: iso3, dtype: object


In [109]:
# classify the missing income classes from: https://datatopics.worldbank.org/world-development-indicators/images/figures-png/world-by-income-sdg-atlas-2018.pdf
sotj_2023.loc[sotj_2023["iso3"] == "AIA","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "ALA","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "ANT","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "ATF","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "BES","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "BLM","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "BVT","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "CCK","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "CXR","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "COK","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "ESH","income_class"] = "Low income"
sotj_2023.loc[sotj_2023["iso3"] == "FLK","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "GGY","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "GLP","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "GUF","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "HMD","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "IOT","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "JEY","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "MSR","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "MTQ","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "MYT","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "NFK","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "NIU","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "TWN","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "PAL","income_class"] = "Upper-middle income"
sotj_2023.loc[sotj_2023["iso3"] == "PCN","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "REU","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "SGS","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "SHN","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "SJM","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "SPM","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "TKL","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "TMP","income_class"] = "Lower-middle income"
sotj_2023.loc[sotj_2023["iso3"] == "UMI","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "TMP","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "VAT","income_class"] = "High income"
sotj_2023.loc[sotj_2023["iso3"] == "XXK","income_class"] = "Upper-middle income"
sotj_2023.loc[sotj_2023["iso3"] == "WLF","income_class"] = "High income"
iso3_with_nan_income = sotj_2023.loc[sotj_2023["income_class"].isna(), "iso3"]
sorted_iso3_with_nan_income = iso3_with_nan_income.sort_values()
print(sorted_iso3_with_nan_income)

252    WSH
Name: iso3, dtype: object


In [110]:
sotj_2023["IncomeClass2"] = sotj_2023["income_class"]
sotj_2023.loc[sotj_2023["income_class"] == "High income", "IncomeClass2"] = "Higher income"
sotj_2023.loc[sotj_2023["income_class"] == "Upper-middle income", "IncomeClass2"] = "Higher income"
sotj_2023.loc[sotj_2023["income_class"] == "Lower-middle income", "IncomeClass2"] = "Lower income"
sotj_2023.loc[sotj_2023["income_class"] == "Low income", "IncomeClass2"] = "Lower income"
sotj_2023["IncomeClass"] = sotj_2023["income_class"]

## File 1: Cluster analysis

In [111]:
def create_group(df_merged2,category):
    
    df_region = df_merged2.groupby(category).sum()

    df_region["OWTE Tax loss incurred (share of global, SOTJ2023, sum)"] = df_region.groupby(category)["OW: Tax revenue loss (% total)"].sum()
    df_region["OWTE Tax loss incurred (USDm, SOTJ2023, sum)"] = df_region.groupby(category)["OW: Tax revenue loss (USD million)"].sum()
    df_region["OWTE Tax loss incurred (share of current tax revenue, SOTJ2023, sum)"] = df_region.groupby(category)['OW: Tax revenue loss (USD million)'].sum()*1000000/df_region.groupby(category)['GRD: Total tax revenue (imp)'].sum()
    df_region["CTA Tax loss incurred (share of global, SOTJ2023, sum)"] = df_region.groupby(category)["Loss TA using CIT (% total)"].sum()
    df_region["CTA Tax loss incurred (USDm, SOTJ2023, sum)"] = df_region.groupby(category)["Loss TA using CIT (USD million)"].sum()
    df_region["CTA Tax loss incurred (share of current tax revenue, SOTJ2023, sum)"] = df_region.groupby(category)["Loss TA using CIT (USD million)"].sum()*1000000/df_region.groupby(category)['GRD: Total tax revenue (imp)'].sum()
    df_region["CTA Profit shifted outward (USDm, SOTJ2023, sum)"] = df_region.groupby(category)["Profit shifted out"].sum()
    df_region["Total tax loss incurred (share of global, SOTJ2023, sum)"] = df_region.groupby(category)["Loss Total using CIT (% total)"].sum()
    df_region["Total tax loss incurred (USDm, SOTJ2023, sum)"] = df_region.groupby(category)["Loss Total using CIT (USD million)"].sum()
    df_region["Total tax loss incurred (share of current tax revenue, SOTJ2023, sum)"] = df_region.groupby(category)["Loss Total using CIT (USD million)"].sum()*1000000/df_region.groupby(category)['GRD: Total tax revenue (imp)'].sum()
    # Filter the DataFrame to include only rows with positive "Loss Total using CIT (USD million)"
    filtered_df = df_region[df_region["Loss Total using CIT (USD million)"] > 0]
    # Filter out rows with NaN values
    filtered_df = filtered_df[~filtered_df["Loss Total using CIT (USD million)"].isna()]
    # Perform the aggregation and calculation, and assign it to the new column
    df_region["Total tax loss incurred (share of current health expenditures, SOTJ2023, sum)"] = ((filtered_df.groupby(category)["Loss Total using CIT (USD million)"].sum() * 1000000) / (filtered_df.groupby(category)['WHO: Government health expenditure'].apply(lambda group: group.sum() if not group.empty else pd.NA)))
    df_region["Total tax loss incurred (share of current GDP, SOTJ2023, sum)"] = df_region.groupby(category)['Loss Total using CIT (USD million)'].sum()*1000000/df_region.groupby(category)['GDP'].sum()

    df_region["OWTE Tax loss inflicted (share of global, SOTJ2023, sum)"] = df_region.groupby(category)["Harm OW: Total (% total)"].sum()/100
    df_region["OWTE Tax loss inflicted (USDm, SOTJ2023, sum)"] = df_region.groupby(category)["Harm OW: Total (USD million)"].sum()
    df_region["CTA Tax loss inflicted (share of global, SOTJ2023, sum)"] = df_region.groupby(category)["Harm TA using CIT (% total)"].sum()
    df_region["CTA Tax loss inflicted (USDm, SOTJ2023, sum)"] = df_region.groupby(category)["Harm TA: Total using CIT (USD million)"].sum()
    df_region["CTA Profit shifted inward (USDm, SOTJ2023, sum)"] = df_region.groupby(category)["Profit shifted in"].sum()
    df_region["CTA Tax revenue gain (USDm, SOTJ2023, sum)"] = df_region.groupby(category)["TA: Tax revenue gain using CIT (USD million)"].sum()
    df_region["Total tax loss inflicted (share of global, SOTJ2023, sum)"] = df_region.groupby(category)["Harm: Total using CIT (% total)"].sum()/100
    df_region["Total tax loss inflicted (USDm, SOTJ2023, sum)"] = df_region.groupby(category)["Harm: Total using CIT (USD million)"].sum()

    df_region["GDP (share of global, sum)"] = df_region.groupby(category)["GDP share"].sum()
    df_region["GDP (sum)"] = df_region.groupby(category)["GDP"].sum()
    df_region["Population (share of global, sum)"] = df_region.groupby(category)["Population share"].sum()
    df_region["Population (sum)"] = df_region.groupby(category)["population"].sum()     
       
    if (len(df_merged2[category].dropna().unique())==2) and (category != "IncomeClass") and (category != "IncomeClass2"):
        
        df_region["Region"] = df_region.index.fillna(0).map(lambda x: "{}{}".format(["Complement of: ",""][int(x)],category))
    else:
        df_region["Region"] = df_region.index
    
    if category == "region_final":
        df_region['"Region" or "Group"?'] = "region"
    else:
        df_region['"Region" or "Group"?'] = "group"
    
    df_region = df_region[['Region','"Region" or "Group"?',
                           "OWTE Tax loss incurred (share of global, SOTJ2023, sum)","OWTE Tax loss incurred (USDm, SOTJ2023, sum)","OWTE Tax loss incurred (share of current tax revenue, SOTJ2023, sum)",
                           "CTA Tax loss incurred (share of global, SOTJ2023, sum)","CTA Tax loss incurred (USDm, SOTJ2023, sum)","CTA Tax loss incurred (share of current tax revenue, SOTJ2023, sum)","CTA Profit shifted outward (USDm, SOTJ2023, sum)",
                           "Total tax loss incurred (share of global, SOTJ2023, sum)","Total tax loss incurred (USDm, SOTJ2023, sum)","Total tax loss incurred (share of current tax revenue, SOTJ2023, sum)","Total tax loss incurred (share of current health expenditures, SOTJ2023, sum)","Total tax loss incurred (share of current GDP, SOTJ2023, sum)",
                           "OWTE Tax loss inflicted (share of global, SOTJ2023, sum)","OWTE Tax loss inflicted (USDm, SOTJ2023, sum)",
                           "CTA Tax loss inflicted (share of global, SOTJ2023, sum)","CTA Tax loss inflicted (USDm, SOTJ2023, sum)","CTA Profit shifted inward (USDm, SOTJ2023, sum)","CTA Tax revenue gain (USDm, SOTJ2023, sum)",
                           "Total tax loss inflicted (share of global, SOTJ2023, sum)","Total tax loss inflicted (USDm, SOTJ2023, sum)",
                           "GDP (share of global, sum)","GDP (sum)","Population (share of global, sum)","Population (sum)"]]

    return df_region

In [112]:
##Cluster analysis

#Define some groups and rename some existing ones to be more descriptive
sotj_2023.rename(columns={"ISO-3 code of country": "country_iso3"}, inplace=True)
sotj_2023["TOTAL"] = 1
sotj_2023["Second empire"] = (sotj_2023["ukt"].astype(bool) | (sotj_2023["country_iso3"] == "GBR"))
sotj_2023["Axis"] = (sotj_2023["country_iso3"].isin(["GBR","NLD","LUX","CHE"])) | sotj_2023["Second empire"]
sotj_2023["EU-28"] = (sotj_2023["eu28"] == 1)
sotj_2023["EU-28 OCTs"] = (sotj_2023["EU28_OCT"] == 1)
sotj_2023["EU-28 + EU-28 OCTs"] = (sotj_2023["eu28"] == 1) | (sotj_2023["EU28_OCT"] == 1) 
sotj_2023["EU-27"] = (sotj_2023["EU27"] == 1)
sotj_2023["EU-27 OCTs"] = (sotj_2023["EU27_OCT"] == 1)
sotj_2023["EU-27 + EU-27 OCTs"] = (sotj_2023["EU27"] == 1) | (sotj_2023["EU27_OCT"] == 1) 
sotj_2023["OECD"] = (sotj_2023["oecd"] == 1)
sotj_2023["OECD OCTs"] = (sotj_2023["oecd_oct"] == 1)
sotj_2023["OECD + OECD OCTs"] = (sotj_2023["oecd"] == 1) | (sotj_2023["oecd_oct"] == 1) 
sotj_2023['Region'] = sotj_2023['region_tjn']
sotj_2023["UK"] = (sotj_2023["country_iso3"] == "GBR")
sotj_2023["UK OCTs"] = (sotj_2023["GBR_OCT"] == 1)
sotj_2023["UK + UK OCTs"] = (sotj_2023["UK"] == 1) | (sotj_2023["GBR_OCT"] == 1) 
sotj_2023["EU blacklist (201006)"] = (sotj_2023["th_eu_blacklist_201006"] == 1)
sotj_2023["EU greylist (201006)"] = (sotj_2023["th_eu_greylist_201006"] == 1)
sotj_2023["UNCTAD (2015) list of tax havens"] = (sotj_2023["th_unctad2015"] == 1)
sotj_2023["Countries with ETR < 10 per cent"] = (sotj_2023["ETR"] < 0.1)

df_merged = sotj_2023.copy()
#Create the groups
df_merged2 = sotj_2023.copy()
df_total_pho = create_group(df_merged2,"TOTAL")
df_income_pho = create_group(df_merged2,"IncomeClass")
df_income_2class_pho = create_group(df_merged2,"IncomeClass2")
df_region_pho = create_group(df_merged2,"Region")
df_eu28_pho = create_group(df_merged2,"EU-28")
df_eu28oct_pho = create_group(df_merged2,"EU-28 OCTs")
df_eu28_and_eu28oct_pho = create_group(df_merged2,"EU-28 + EU-28 OCTs")
df_eu27_pho = create_group(df_merged2,"EU-27")
df_eu27oct_pho = create_group(df_merged2,"EU-27 OCTs")
df_eu27_and_eu27oct_pho = create_group(df_merged2,"EU-27 + EU-27 OCTs")
df_oecd_pho = create_group(df_merged2,"OECD")
df_oecdoct_pho = create_group(df_merged2,"OECD OCTs")
df_oecd_and_oecdoct_pho = create_group(df_merged2,"OECD + OECD OCTs")
df_uk_pho = create_group(df_merged2,"UK")
df_ukoct_pho = create_group(df_merged2,"UK OCTs")
df_uk_and_ukoct_pho = create_group(df_merged2,"UK + UK OCTs")
#df_g24 = create_group(df_merged2,"g24")
df_spider_pho = create_group(df_merged2,"Second empire")
df_axis_pho = create_group(df_merged2,"Axis")
df_EUblack_pho = create_group(df_merged2,"EU blacklist (201006)")
df_EUgrey_pho = create_group(df_merged2,"EU greylist (201006)")
df_UNCTADlist_pho = create_group(df_merged2,"UNCTAD (2015) list of tax havens")
#df_trans_rating_pho = create_group(df_merged2,"trans_rating_oecd")
#df_harmful_regimes_pho = create_group(df_merged2,"harmful_regimes_oecd")
df_etr_below10_pho = create_group(df_merged2,"Countries with ETR < 10 per cent")

df_groups = pd.concat([df_total_pho,df_income_2class_pho,df_income_pho,df_region_pho,df_eu28_pho,df_eu28oct_pho,df_eu28_and_eu28oct_pho,df_eu27_pho,df_eu27oct_pho,df_eu27_and_eu27oct_pho,df_uk_pho,df_ukoct_pho,df_uk_and_ukoct_pho,df_oecd_pho,df_oecdoct_pho,df_oecd_and_oecdoct_pho,df_spider_pho,df_axis_pho,df_EUblack_pho,df_EUgrey_pho,df_UNCTADlist_pho,df_etr_below10_pho])
df_groups = df_groups[['Region','"Region" or "Group"?',
                           "OWTE Tax loss incurred (share of global, SOTJ2023, sum)","OWTE Tax loss incurred (USDm, SOTJ2023, sum)","OWTE Tax loss incurred (share of current tax revenue, SOTJ2023, sum)",
                           "CTA Tax loss incurred (share of global, SOTJ2023, sum)","CTA Tax loss incurred (USDm, SOTJ2023, sum)","CTA Tax loss incurred (share of current tax revenue, SOTJ2023, sum)","CTA Profit shifted outward (USDm, SOTJ2023, sum)",
                           "Total tax loss incurred (share of global, SOTJ2023, sum)","Total tax loss incurred (USDm, SOTJ2023, sum)","Total tax loss incurred (share of current tax revenue, SOTJ2023, sum)","Total tax loss incurred (share of current health expenditures, SOTJ2023, sum)","Total tax loss incurred (share of current GDP, SOTJ2023, sum)",
                           "OWTE Tax loss inflicted (share of global, SOTJ2023, sum)","OWTE Tax loss inflicted (USDm, SOTJ2023, sum)",
                           "CTA Tax loss inflicted (share of global, SOTJ2023, sum)","CTA Tax loss inflicted (USDm, SOTJ2023, sum)","CTA Profit shifted inward (USDm, SOTJ2023, sum)","CTA Tax revenue gain (USDm, SOTJ2023, sum)",
                           "Total tax loss inflicted (share of global, SOTJ2023, sum)","Total tax loss inflicted (USDm, SOTJ2023, sum)",
                           "GDP (share of global, sum)","GDP (sum)","Population (share of global, sum)","Population (sum)"]]
df_groups = df_groups.rename(columns={'Region': 'Group'})

From the below, only Files 1 and 2 are produced so far for SOTJ2023

In [113]:
##FILE 1 - INTERNAL TJN ANALYSIS FILE
writer = pd.ExcelWriter(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Scale of Tax Injustice/State of Tax Justice report/2023 Report/230808_file1_sotj_2023_analysis_old.xlsx', engine='xlsxwriter')
workbook  = writer.book

#Define formats
cell_format_num2 = workbook.add_format({'align':'center','num_format': '#,##0.0'})
cell_format_num3 = workbook.add_format({'align':'center','num_format': '#,##0'})
cell_format_pct_leftborder = workbook.add_format({'align':'center','num_format':10,'left':True})
cell_format_num2_leftborder = workbook.add_format({'align':'center','num_format':'#,##0.0','left':True})
cell_format_pct = workbook.add_format({'num_format':10})
cell_format_pct_rightborder = workbook.add_format({'align':'right','num_format':10,'right':True})
cell_format_rightborder = workbook.add_format({'right':True})
cell_format_header = workbook.add_format({'bold': True,'text_wrap': True,'valign': 'top','fg_color': '#2586C6','bottom': True})
cell_format_wrap = workbook.add_format({'text_wrap': True})

#Cluster analysis
df_groups.loc[(df_groups["Group"] == 'TOTAL'), "Group"] = 'TOTAL'
df_groups.loc[(df_groups["Group"] == 'Higher income'), "Group"] = 'Higher income (1/2)'
df_groups.loc[(df_groups["Group"] == 'Lower income'), "Group"] = 'Lower income (2/2)'
df_groups.loc[(df_groups["Group"] == 'High income'), "Group"] = 'High income (1/4)'
df_groups.loc[(df_groups["Group"] == 'Low income'), "Group"] = 'Low income (2/4)'
df_groups.loc[(df_groups["Group"] == 'Lower middle income'), "Group"] = 'Lower middle income (3/4)'
df_groups.loc[(df_groups["Group"] == 'Upper middle income'), "Group"] = 'Upper middle income (4/4)'
df_groups.loc[(df_groups["Group"] == ''), "Group"] = 'Not rated'
df_groups.loc[(df_groups["Group"] == 'Not rated'), '"Region" or "Group"?'] = 'OECD transparency rating'
df_groups.loc[(df_groups["Group"] == 'Compliant'), '"Region" or "Group"?'] = 'OECD transparency rating'
df_groups.loc[(df_groups["Group"] == 'Largely Compliant'), '"Region" or "Group"?'] = 'OECD transparency rating'
df_groups.loc[(df_groups["Group"] == 'Non-Compliant'), '"Region" or "Group"?'] = 'OECD transparency rating'
df_groups.loc[(df_groups["Group"] == 'Partially Compliant'), '"Region" or "Group"?'] = 'OECD transparency rating'
df_groups.loc[(df_groups["Group"] == 'Provisionally Largely Compliant'), '"Region" or "Group"?'] = 'OECD transparency rating'
df_groups.loc[(df_groups["Group"] == 'Harmful'), '"Region" or "Group"?'] = 'OECD harmful regimes'
df_groups.loc[(df_groups["Group"] == 'Not harmful'), '"Region" or "Group"?'] = 'OECD harmful regimes'
df_groups.loc[(df_groups["Group"] == 'Under review'), '"Region" or "Group"?'] = 'OECD harmful regimes'
df_groups.to_excel(writer, sheet_name='Cluster analysis', index=False)
worksheet = writer.sheets['Cluster analysis']
worksheet.set_column('A:A', 26, None)
worksheet.freeze_panes(1, 2)
columns = ['C','F','J','O','Q','U','W']
for column in columns:
    worksheet.set_column(column+':'+column, None, cell_format_pct_leftborder)
columns = ['E','H','L','M','N','Y']
for column in columns:
    worksheet.set_column(column+':'+column, None, cell_format_pct)
columns = ['D','G','I','K','P','R','S','T','V','X','Z']
for column in columns:
    worksheet.set_column(column+':'+column, None, cell_format_num2)

#writer.save()
writer.close()

In [114]:
##EXCEL 2 - SUMMARY RESULTS TABLE FOR THE PRESS

writer = pd.ExcelWriter(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Scale of Tax Injustice/State of Tax Justice report/2023 Report/230808_file2_sotj_2023_press_old.xlsx', engine='xlsxwriter')
workbook  = writer.book

sotj_2023_summaryResults = sotj_2023.copy()

# Total annual tax loss (USD million) 
#Total annual tax loss (% of GDP)	
#Total annual tax loss (% of health expenditures)	 
#Of which: Corporate tax abuse (USD million) 	 
#Of which: Offshore wealth (USD million) 	 
#Full vaccinations possible (millions) 	
#Full vaccinations possible: Share of population	 
#Total inflicted tax loss 	 
#Of which: Tax loss inflicted on others via corporate tax abuse (USD million) 	 
#Of which: Tax loss inflicted on others via offshore wealth tax evasion (USD million) 	
#Share of total global inflicted tax loss	 
#Total inflicted tax loss in terms of vaccines (millions) 


#Define formats
cell_format_num2 = workbook.add_format({'align':'center','num_format': '#,##0.0'})
cell_format_num3 = workbook.add_format({'align':'center','num_format': '#,##0'})
cell_format_pct_leftborder = workbook.add_format({'align':'center','num_format':10,'left':True})
cell_format_num2_leftborder = workbook.add_format({'align':'center','num_format':'#,##0.0','left':True})
cell_format_pct = workbook.add_format({'num_format':10})
cell_format_pct_rightborder = workbook.add_format({'align':'right','num_format':10,'right':True})
cell_format_rightborder = workbook.add_format({'right':True})
cell_format_header = workbook.add_format({'bold': True,'text_wrap': True,'valign': 'top','fg_color': '#2586C6','bottom': True})
cell_format_wrap = workbook.add_format({'text_wrap': True})

df_groups.to_excel(writer, sheet_name='SOTJ23 - Summary of results', index=False)
worksheet = writer.sheets['SOTJ23 - Summary of results']
worksheet.set_column('A:A', 26, None)
worksheet.freeze_panes(1, 1)
columns = ['C','F','J','O','Q','U','W']
for column in columns:
    worksheet.set_column(column+':'+column, None, cell_format_pct_leftborder)
columns = ['E','H','L','M','N','Y']
for column in columns:
    worksheet.set_column(column+':'+column, None, cell_format_pct)
columns = ['D','G','I','K','P','R','S','T','V','X','Z']
for column in columns:
    worksheet.set_column(column+':'+column, None, cell_format_num2)

#writer.save()
writer.close()



In [115]:
##FILE 3 COUNTRY LEVEL RESULTS
writer = pd.ExcelWriter(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Scale of Tax Injustice/State of Tax Justice report/2023 Report/230808_country_level_old.xlsx', engine='xlsxwriter')
workbook  = writer.book

#Define formats
cell_format_num2 = workbook.add_format({'align':'center','num_format': '#,##0.0'})
cell_format_num3 = workbook.add_format({'align':'center','num_format': '#,##0'})
cell_format_pct_leftborder = workbook.add_format({'align':'center','num_format':10,'left':True})
cell_format_num2_leftborder = workbook.add_format({'align':'center','num_format':'#,##0.0','left':True})
cell_format_pct = workbook.add_format({'num_format':10})
cell_format_pct_rightborder = workbook.add_format({'align':'right','num_format':10,'right':True})
cell_format_rightborder = workbook.add_format({'right':True})
cell_format_header = workbook.add_format({'bold': True,'text_wrap': True,'valign': 'top','fg_color': '#2586C6','bottom': True})
cell_format_wrap = workbook.add_format({'text_wrap': True})


df_countries = create_group(df_merged2,"country_iso3")
df_countries = df_countries[['Region','"Region" or "Group"?',"OWTE Tax loss incurred (share of global, SOTJ2023, sum)","OWTE Tax loss incurred (USDm, SOTJ2023, sum)","OWTE Tax loss incurred (share of current tax revenue, SOTJ2023, sum)",
                           "CTA Tax loss incurred (share of global, SOTJ2023, sum)","CTA Tax loss incurred (USDm, SOTJ2023, sum)","CTA Tax loss incurred (share of current tax revenue, SOTJ2023, sum)","CTA Profit shifted outward (USDm, SOTJ2023, sum)",
                           "Total tax loss incurred (share of global, SOTJ2023, sum)","Total tax loss incurred (USDm, SOTJ2023, sum)","Total tax loss incurred (share of current tax revenue, SOTJ2023, sum)","Total tax loss incurred (share of current health expenditures, SOTJ2023, sum)","Total tax loss incurred (share of current GDP, SOTJ2023, sum)",
                           "OWTE Tax loss inflicted (share of global, SOTJ2023, sum)","OWTE Tax loss inflicted (USDm, SOTJ2023, sum)",
                           "CTA Tax loss inflicted (share of global, SOTJ2023, sum)","CTA Tax loss inflicted (USDm, SOTJ2023, sum)","CTA Profit shifted inward (USDm, SOTJ2023, sum)","CTA Tax revenue gain (USDm, SOTJ2023, sum)",
                           "Total tax loss inflicted (share of global, SOTJ2023, sum)","Total tax loss inflicted (USDm, SOTJ2023, sum)",
                           "GDP (share of global, sum)","GDP (sum)","Population (share of global, sum)","Population (sum)"]]

df_countries.to_excel(writer, sheet_name='Country results', index=False)
worksheet = writer.sheets['Country results']
worksheet.set_column('A:A', 26, None)
worksheet.freeze_panes(1, 2)
columns = ['C','F','J','O','Q','U','W']
for column in columns:
    worksheet.set_column(column+':'+column, None, cell_format_pct_leftborder)
columns = ['E','H','L','M','N','Y']
for column in columns:
    worksheet.set_column(column+':'+column, None, cell_format_pct)
columns = ['D','G','I','K','P','R','S','T','V','X','Z']
for column in columns:
    worksheet.set_column(column+':'+column, None, cell_format_num2)

#writer.save()
writer.close()

ENDS HERE - the below we might use in the future to adjust for the purposes of SOTJ2023, but don't run it now.

In [116]:
STOP HERE
#Sheet 1: CTHI2021 short
cthi_2021_short = cthi[['country','country_iso2','country_iso3',"income_final_2018","region_final","EU-28","EU-28 OCTs","EU-27","EU-27 OCTs","UK OCTs","OECD","OECD OCTs",'cthi_2021_rank','cthi_2021','cthi_2021_share','cthi_2021_gsw','cthi_2021_hs']]
cthi_2021_short_sorted = cthi_2021_short.sort_values(by=['cthi_2021'], ascending=False)
cthi_2021_short_sorted.to_excel(writer, sheet_name='CTHI2021 short', index=False)
worksheet = writer.sheets['CTHI2021 short']
worksheet.set_column('A:A', 26, None)
worksheet.set_column('B:L', None, None, {'hidden':True})
worksheet.set_column('M:Q', 15, None)
worksheet.freeze_panes(1, 0)

#Sheet 2: CTHI2021 full
cthi_2021_full = cthi[['country','country_iso2','country_iso3',"income_final_2018","region_final","EU-28","EU-28 OCTs","EU-27","EU-27 OCTs","UK OCTs","OECD","OECD OCTs",'cthi_2021_rank','cthi_2021','cthi_2021_share','cthi_2021_gsw','cthi_2021_hs','cthi_2021_hs_hi1','cthi_2021_hs_hi2','cthi_2021_hs_hi3','cthi_2021_hs_hi4','cthi_2021_hs_hi5','cthi_2021_hs_hi6','cthi_2021_hs_hi7','cthi_2021_hs_hi8','cthi_2021_hs_hi9','cthi_2021_hs_hi10','cthi_2021_hs_hi11','cthi_2021_hs_hi12','cthi_2021_hs_hi13','cthi_2021_hs_hi14','cthi_2021_hs_hi15','cthi_2021_hs_hi16','cthi_2021_hs_hi17','cthi_2021_hs_hi18','cthi_2021_hs_hi19','cthi_2021_hs_hi20','cthi_2021_hs_cat1','cthi_2021_hs_cat2','cthi_2021_hs_cat3','cthi_2021_hs_cat4','cthi_2021_hs_cat5','tot_cy_fdi_in_final_2019', 'tot_cy_fdi_out_final_2019','suminout_fdi_final_2019','tot_y_fdi_final_2019']]
cthi_2021_full_sorted = cthi_2021_full.sort_values(by=['cthi_2021'], ascending=False)
cthi_2021_full_sorted.to_excel(writer, sheet_name='CTHI2021 full', index=False)
worksheet = writer.sheets['CTHI2021 full']
worksheet.set_column('A:A', 26, None)
worksheet.set_column('B:L', None, None, {'hidden':True})
worksheet.freeze_panes(1, 1)

#Sheet 3: CTHI2021 and CTHI2019 full
cthi_2019_2021_full = cthi[['country','country_iso2','country_iso3',"income_final_2018","region_final","EU-28","EU-28 OCTs","EU-27","EU-27 OCTs","UK OCTs","OECD","OECD OCTs",'cthi_2021_rank','cthi_2021','cthi_2021_share','cthi_2021_gsw','cthi_2021_hs','cthi_2021_hs_hi1','cthi_2021_hs_hi2','cthi_2021_hs_hi3','cthi_2021_hs_hi4','cthi_2021_hs_hi5','cthi_2021_hs_hi6','cthi_2021_hs_hi7','cthi_2021_hs_hi8','cthi_2021_hs_hi9','cthi_2021_hs_hi10','cthi_2021_hs_hi11','cthi_2021_hs_hi12','cthi_2021_hs_hi13','cthi_2021_hs_hi14','cthi_2021_hs_hi15','cthi_2021_hs_hi16','cthi_2021_hs_hi17','cthi_2021_hs_hi18','cthi_2021_hs_hi19','cthi_2021_hs_hi20','cthi_2021_hs_cat1','cthi_2021_hs_cat2','cthi_2021_hs_cat3','cthi_2021_hs_cat4','cthi_2021_hs_cat5','cthi_2019','cthi_2019_gsw','cthi_2019_hs','cthi_2019_hs_hi1','cthi_2019_hs_hi2','cthi_2019_hs_hi3','cthi_2019_hs_hi4','cthi_2019_hs_hi5','cthi_2019_hs_hi6','cthi_2019_hs_hi7','cthi_2019_hs_hi8','cthi_2019_hs_hi9','cthi_2019_hs_hi10','cthi_2019_hs_hi11','cthi_2019_hs_hi12','cthi_2019_hs_hi13','cthi_2019_hs_hi14','cthi_2019_hs_hi15','cthi_2019_hs_hi16','cthi_2019_hs_hi17','cthi_2019_hs_hi18','cthi_2019_hs_hi19','cthi_2019_hs_hi20','cthi_2019_hs_cat1','cthi_2019_hs_cat2','cthi_2019_hs_cat3','cthi_2019_hs_cat4','cthi_2019_hs_cat5']]
cthi_2019_2021_full_sorted = cthi_2019_2021_full.sort_values(by=['cthi_2021'], ascending=False)
cthi_2019_2021_full_sorted.to_excel(writer, sheet_name='CTHI2021 and CTHI2019 full', index=False)
worksheet = writer.sheets['CTHI2021 and CTHI2019 full']
worksheet.set_column('A:A', 26, None)
worksheet.set_column('B:L', None, None, {'hidden':True})
worksheet.freeze_panes(1, 1)

#Sheet 4: Changes 2019-2021
changes = cthi[['country','country_iso2','country_iso3',"income_final_2018","region_final","EU-28","EU-28 OCTs","EU-27","EU-27 OCTs","UK OCTs","OECD","OECD OCTs",'included_in_cthi_2021','included_in_cthi_2019','cthi_2021','cthi_2019','cthi_2021 - cthi_2019','cthi_2021 - cthi_2019 (%)','cthi_2021_rank','cthi_2019_rank','cthi_2021_rank - cthi_2019_rank','cthi_2021_rank - cthi_2019_rank (%)','cthi_2021_share','cthi_2019_share','cthi_2021_share - cthi_2019_share','cthi_2021_share - cthi_2019_share (%)','cthi_2021_gsw','cthi_2019_gsw','cthi_2021_gsw - cthi_2019_gsw','cthi_2021_gsw - cthi_2019_gsw (%)', 'cthi_2021_hs','cthi_2019_hs','cthi_2021_hs - cthi_2019_hs','cthi_2021_hs - cthi_2019_hs (%)', 'cthi_2021_hs_cat1','cthi_2019_hs_cat1','cthi_2021_hs_cat1 - cthi_2019_hs_cat1','cthi_2021_hs_cat1 - cthi_2019_hs_cat1 (%)', 'cthi_2021_hs_cat2','cthi_2019_hs_cat2','cthi_2021_hs_cat2 - cthi_2019_hs_cat2','cthi_2021_hs_cat2 - cthi_2019_hs_cat2 (%)', 'cthi_2021_hs_cat3','cthi_2019_hs_cat3','cthi_2021_hs_cat3 - cthi_2019_hs_cat3','cthi_2021_hs_cat3 - cthi_2019_hs_cat3 (%)', 'cthi_2021_hs_cat4','cthi_2019_hs_cat4','cthi_2021_hs_cat4 - cthi_2019_hs_cat4','cthi_2021_hs_cat4 - cthi_2019_hs_cat4 (%)', 'cthi_2021_hs_cat5','cthi_2019_hs_cat5','cthi_2021_hs_cat5 - cthi_2019_hs_cat5','cthi_2021_hs_cat5 - cthi_2019_hs_cat5 (%)', 'cthi_2021_hs_hi1','cthi_2019_hs_hi1','cthi_2021_hs_hi1 - cthi_2019_hs_hi1','cthi_2021_hs_hi1 - cthi_2019_hs_hi1 (%)', 'cthi_2021_hs_hi2','cthi_2019_hs_hi2','cthi_2021_hs_hi2 - cthi_2019_hs_hi2','cthi_2021_hs_hi2 - cthi_2019_hs_hi2 (%)', 'cthi_2021_hs_hi3','cthi_2019_hs_hi3','cthi_2021_hs_hi3 - cthi_2019_hs_hi3','cthi_2021_hs_hi3 - cthi_2019_hs_hi3 (%)', 'cthi_2021_hs_hi4','cthi_2019_hs_hi4','cthi_2021_hs_hi4 - cthi_2019_hs_hi4','cthi_2021_hs_hi4 - cthi_2019_hs_hi4 (%)', 'cthi_2021_hs_hi5','cthi_2019_hs_hi5','cthi_2021_hs_hi5 - cthi_2019_hs_hi5','cthi_2021_hs_hi5 - cthi_2019_hs_hi5 (%)', 'cthi_2021_hs_hi6','cthi_2019_hs_hi6','cthi_2021_hs_hi6 - cthi_2019_hs_hi6','cthi_2021_hs_hi6 - cthi_2019_hs_hi6 (%)', 'cthi_2021_hs_hi7','cthi_2019_hs_hi7','cthi_2021_hs_hi7 - cthi_2019_hs_hi7','cthi_2021_hs_hi7 - cthi_2019_hs_hi7 (%)', 'cthi_2021_hs_hi8','cthi_2019_hs_hi8','cthi_2021_hs_hi8 - cthi_2019_hs_hi8','cthi_2021_hs_hi8 - cthi_2019_hs_hi8 (%)', 'cthi_2021_hs_hi9','cthi_2019_hs_hi9','cthi_2021_hs_hi9 - cthi_2019_hs_hi9','cthi_2021_hs_hi9 - cthi_2019_hs_hi9 (%)', 'cthi_2021_hs_hi10','cthi_2019_hs_hi10','cthi_2021_hs_hi10 - cthi_2019_hs_hi10','cthi_2021_hs_hi10 - cthi_2019_hs_hi10 (%)', 'cthi_2021_hs_hi11','cthi_2019_hs_hi11','cthi_2021_hs_hi11 - cthi_2019_hs_hi11','cthi_2021_hs_hi11 - cthi_2019_hs_hi11 (%)', 'cthi_2021_hs_hi12','cthi_2019_hs_hi12','cthi_2021_hs_hi12 - cthi_2019_hs_hi12','cthi_2021_hs_hi12 - cthi_2019_hs_hi12 (%)', 'cthi_2021_hs_hi13','cthi_2019_hs_hi13','cthi_2021_hs_hi13 - cthi_2019_hs_hi13','cthi_2021_hs_hi13 - cthi_2019_hs_hi13 (%)', 'cthi_2021_hs_hi14','cthi_2019_hs_hi14','cthi_2021_hs_hi14 - cthi_2019_hs_hi14','cthi_2021_hs_hi14 - cthi_2019_hs_hi14 (%)', 'cthi_2021_hs_hi15','cthi_2019_hs_hi15','cthi_2021_hs_hi15 - cthi_2019_hs_hi15','cthi_2021_hs_hi15 - cthi_2019_hs_hi15 (%)', 'cthi_2021_hs_hi16','cthi_2019_hs_hi16','cthi_2021_hs_hi16 - cthi_2019_hs_hi16','cthi_2021_hs_hi16 - cthi_2019_hs_hi16 (%)', 'cthi_2021_hs_hi17','cthi_2019_hs_hi17','cthi_2021_hs_hi17 - cthi_2019_hs_hi17','cthi_2021_hs_hi17 - cthi_2019_hs_hi17 (%)', 'cthi_2021_hs_hi18','cthi_2019_hs_hi18','cthi_2021_hs_hi18 - cthi_2019_hs_hi18','cthi_2021_hs_hi18 - cthi_2019_hs_hi18 (%)', 'cthi_2021_hs_hi19','cthi_2019_hs_hi19','cthi_2021_hs_hi19 - cthi_2019_hs_hi19','cthi_2021_hs_hi19 - cthi_2019_hs_hi19 (%)', 'cthi_2021_hs_hi20','cthi_2019_hs_hi20','cthi_2021_hs_hi20 - cthi_2019_hs_hi20','cthi_2021_hs_hi20 - cthi_2019_hs_hi20 (%)']]
changes = changes[~(changes['cthi_2021_gsw - cthi_2019_gsw'].isnull())]
changes_sorted = changes.sort_values(by=['cthi_2021'], ascending=False)
changes_sorted.to_excel(writer, sheet_name='Changes 2019-2021', index=False)
worksheet = writer.sheets['Changes 2019-2021']
worksheet.set_column('A:A', 26, None)
worksheet.set_column('B:N', None, None, {'hidden':True})
columns = ['O','P','Q','R','S','T','U','V','W','X','Y','Z','AA','AB','AC','AD','AE','AF','AG','AH','AI','AJ','AK','AL','AM','AN','AO','AP','AQ','AR','AS','AT','AU','AV','AW','AX','AY','AZ','BA','BB','BC','BD','BE','BF','BG','BH','BI','BJ','BK','BL','BM','BN','BO','BP','BQ','BR','BS','BT','BU','BV','BW','BX','BY','BZ','CA','CB','CC','CD','CE','CF','CG','CH','CI','CJ','CK','CL','CM','CN','CO','CP','CQ','CR','CS','CT','CU','CV','CW','CX','CY','CZ','DA','DB','DC','DD','DE','DF','DG','DH','DI','DJ','DK','DL','DM','DN','DO','DP','DQ','DR','DS','DT','DU','DV','DW','DX','DY','DZ','EA','EB','EC','ED']
for column in columns:
    worksheet.conditional_format(column+'2:'+column+'280', {'type': '3_color_scale', 'min_color': '#6BACD1', 'mid_color': '#FEED7B', 'max_color': '#C94741'})
    worksheet.set_column(column+':'+column, 10, cell_format_num2)
columns = ['S','T','U','V']
for column in columns:
    worksheet.conditional_format(column+'2:'+column+'280', {'type': '3_color_scale', 'min_color': '#C94741', 'mid_color': '#FEED7B', 'max_color': '#6BACD1'})
    worksheet.set_column(column+':'+column, 10, cell_format_num3)
columns = ['W','X','Y','Z','AA','AB','AC','AD']
for column in columns:
    worksheet.set_column(column+':'+column, 10, cell_format_pct)
columns = ['R','V','Z','AD','AH','AL','AP','AT','AX','BB','BF','BJ','BN','BR','BV','BZ','CD','CH','CL','CP','CT','CX','DB','DF','DJ','DN','DR','DV','DZ','ED']
for column in columns:
    worksheet.set_column(column+':'+column, 10, cell_format_pct_rightborder)
worksheet.set_column('O:O', 10, cell_format_num2_leftborder)
worksheet.freeze_panes(1, 1)

#Sheet 5: GSW historical data
cthi_gsw_historical = pd.merge(cthi_gsw,cthi[['country_iso3','included_in_cthi_2019','country','included_in_cthi_2021','gdp_final']],on=['country_iso3'],how='inner')
cthi_gsw_historical = cthi_gsw_historical.loc[cthi_gsw_historical['year'] != 2020]
cthi_gsw_historical = cthi_gsw_historical[['country','country_iso3','year','included_in_cthi_2019','included_in_cthi_2021','cthi_gsw','tot_cy_fdi_in_final','tot_cy_fdi_out_final','suminout_fdi_final','tot_y_fdi_final','gdp_final']]
cthi_gsw_historical.to_excel(writer, sheet_name='GSW historical data', index=False)
worksheet = writer.sheets['GSW historical data']
worksheet.set_column('A:A', 26, None)
worksheet.freeze_panes(1, 3)

#Sheet 6: Rank changes breakdown
cthi['Total change in rank'] = cthi['cthi_2021_rank - cthi_2019_rank']
rank_changes_breakdown = cthi[['country','country_iso2','country_iso2','CTHI: old GSW, old HS, old countries','CTHI rank: old GSW, old HS, old countries','CTHI: new GSW, old HS, old countries','CTHI rank: new GSW, old HS, old countries','CTHI: new GSW, new HS, old countries','CTHI rank: new GSW, new HS, old countries','CTHI: new GSW, new HS, new countries','CTHI rank: new GSW, new HS, new countries','Total change in rank','Rank change due to GSW','Rank change due to HS','Rank change due to new countries']]
rank_changes_breakdown = rank_changes_breakdown.loc[~(rank_changes_breakdown['CTHI: new GSW, new HS, new countries'].isna())]
rank_changes_breakdown_sorted = rank_changes_breakdown.sort_values(by=['CTHI: new GSW, new HS, new countries'], ascending=False)
rank_changes_breakdown_sorted.to_excel(writer, sheet_name='Rank changes breakdown', index=False)
worksheet = writer.sheets['Rank changes breakdown']
worksheet.set_column('A:A', 26, None)
worksheet.set_column('L:O', 26, None)
worksheet.freeze_panes(1, 1)
#columns = ['D','E','F','G','H','I','J','K','L','M','N','O','P','Q']
#for column in columns:
#    worksheet.set_column(column+':'+column, 12, cell_format_num3)
#columns = ['D','F','I','L','O']
#for column in columns:
#    worksheet.set_column(column+':'+column, 12, cell_format_num2)
#columns = ['H','K','N','Q']
#for column in columns:
#    worksheet.set_column(column+':'+column, 12, cell_format_rightborder)
#columns = ['R','S','T']
#for column in columns:
 #   worksheet.set_column(column+':'+column, 12, cell_format_pct)

 #Sheet 8: Country characteristics
country_char = cthi[['country','country_iso2','country_iso3','All CTHI 2021 jurisdictions',"All CTHI 2019 jurisdictions","income_final_2018","IncomeClass2","region_final","EU-28","EU-28 OCTs","EU-28 + EU-28 OCTs","EU-27","EU-27 OCTs","EU-27 + EU-27 OCTs","UK","UK OCTs","UK + UK OCTs","OECD","OECD OCTs","OECD + OECD OCTs","UK","G20","Spider","Axis","Top 3","Top 5","Top 10","Top 15","EU blacklist (201006)","EU greylist (201006)","UNCTAD (2015) list of tax havens","trans_rating_oecd","harmful_regimes_oecd"]]
country_char[["G20"]].astype(bool)
country_char_sorted = country_char.sort_values(by=['country'], ascending=False)
country_char_sorted.to_excel(writer, sheet_name='Country characteristics', index=False)
worksheet = writer.sheets['Country characteristics']
worksheet.set_column('A:A', 26, None)
worksheet.freeze_panes(1, 3)

SyntaxError: invalid syntax (1464160519.py, line 1)

ROBUSTNESS CHECKS

In [ ]:
def create_gsw_alt(year=2019):
    ##Global Scale Weights for each year 
    cthi_gsw = pd.read_stata(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/GSW/20201231 CDIS data.dta')
    cthi_gsw = cthi_gsw[['country','country_iso3','counterpart_country','counterpart_country_iso3','year','cdis17','cdis18','cdis37','cdis38']]
    print(cthi_gsw.dropna(subset=["cdis17","cdis18","cdis37","cdis38"],how="all")["year"].value_counts())
    cthi_gsw = cthi_gsw.loc[cthi_gsw["year"]==year]
    ##Calculate GSW
    #Step 1: taking maximum of reported, derived and zero
    cthi_gsw['zero'] = 0
    cthi_gsw['fdi_in_final'] = cthi_gsw[["cdis17", "cdis18", "zero"]].max(axis=1)
    cthi_gsw['fdi_out_final'] = cthi_gsw[["cdis37", "cdis38", "zero"]].max(axis=1)
    for var in ["cdis17","cdis18","cdis37","cdis38"]:
        cthi_gsw.loc[cthi_gsw[var]<0,var] = 0

    #Step 2: summing by country and year
    cthi_gsw = cthi_gsw.groupby('country_iso3').sum().reset_index()

    #Step 3: summing inward and outward
    cthi_gsw['cthi_gsw_B0'] = cthi_gsw['fdi_in_final'] + cthi_gsw['fdi_out_final']
    cthi_gsw['cthi_gsw_B1'] = cthi_gsw['fdi_in_final'] 
    cthi_gsw['cthi_gsw_B2'] = cthi_gsw['fdi_out_final'] 
    cthi_gsw['cthi_gsw_B3'] = cthi_gsw['cdis18'] + cthi_gsw['cdis38']
    cthi_gsw['cthi_gsw_B4'] = cthi_gsw['cdis17'] + cthi_gsw['cdis37']

    return cthi_gsw

In [ ]:
##Robustness checks
robustness_checks = cthi.copy()
robustness_checks = robustness_checks[~(robustness_checks['cthi_2021'].isnull())]
robustness_checks["log_cthi_2021_gsw"] = np.log(robustness_checks['cthi_2021_gsw'])
robustness_checks["log_cthi_2021_gsw_rescaled"] = 10* ((robustness_checks['log_cthi_2021_gsw'] - robustness_checks['log_cthi_2021_gsw'].min()) / (robustness_checks['log_cthi_2021_gsw'].max() - robustness_checks['log_cthi_2021_gsw'].min()))
robustness_checks[["country_iso3","log_cthi_2021_gsw","log_cthi_2021_gsw_rescaled"]]
cthi_gsw_robust = create_gsw_alt(2019)
cthi_gsw_robust.head()

robustness_checks = cthi.copy()
robustness_checks = robustness_checks[~(robustness_checks['cthi_2021'].isnull())]

robustness_checks['cthi_2021_rcA0'] = (robustness_checks['cthi_2021_hs']**3 * robustness_checks['cthi_2021_gsw']**(1/3))/100
robustness_checks['cthi_2021_def_rcA0'] = "Original CTHI"
#rcA1
robustness_checks['cthi_2021_rcA1'] = (robustness_checks['cthi_2021_hs'] * robustness_checks['cthi_2021_gsw'])/100
robustness_checks['cthi_2021_def_rcA1'] = "(HS * GSW)/100"
#rcA2
robustness_checks['cthi_2021_rcA2'] = (robustness_checks['cthi_2021_hs']**2 * robustness_checks['cthi_2021_gsw']**(1/2))/100
robustness_checks['cthi_2021_def_rcA2'] = "(HS^2 * GSW^(1/2))/100"
#rcA3
robustness_checks['cthi_2021_rcA3'] = (robustness_checks['cthi_2021_hs']**4 * robustness_checks['cthi_2021_gsw']**(1/4))/100
robustness_checks['cthi_2021_def_rcA3'] = "(HS^4 * GSW^(1/4))/100"
#rcA4
robustness_checks['cthi_2021_rcA4'] = (robustness_checks['cthi_2021_hs']**5 * robustness_checks['cthi_2021_gsw']**(1/5))/100
robustness_checks['cthi_2021_def_rcA4'] = "(HS^5 * GSW^(1/5))/100"
#rcA5
robustness_checks['cthi_2021_hs_rescaled'] = 10 * ((robustness_checks['cthi_2021_hs'] - robustness_checks['cthi_2021_hs'].min()) / (robustness_checks['cthi_2021_hs'].max() - robustness_checks['cthi_2021_hs'].min()))
robustness_checks['cthi_2021_gsw_rcA5'] = 10 * ((robustness_checks['cthi_2021_gsw'] - robustness_checks['cthi_2021_gsw'].min()) / (robustness_checks['cthi_2021_gsw'].max() - robustness_checks['cthi_2021_gsw'].min()))
robustness_checks['cthi_2021_rcA5'] = (robustness_checks['cthi_2021_hs_rescaled'] * robustness_checks['cthi_2021_gsw_rcA5'])
robustness_checks['cthi_2021_def_rcA5'] = "HS and GSW scaled to [0,10], then HS * GSW"
#rcA6
robustness_checks["log_cthi_2021_gsw"] = np.log(robustness_checks['cthi_2021_gsw'])
robustness_checks["log_cthi_2021_gsw_rescaled"] = 10* ((robustness_checks['log_cthi_2021_gsw'] - robustness_checks['log_cthi_2021_gsw'].min()) / (robustness_checks['log_cthi_2021_gsw'].max() - robustness_checks['log_cthi_2021_gsw'].min()))
robustness_checks['cthi_2021_rcA6'] = robustness_checks['cthi_2021_hs_rescaled'] * robustness_checks['log_cthi_2021_gsw_rescaled']
robustness_checks['cthi_2021_def_rcA6'] = "HS and log(GSW) scaled to [0,10], then HS * GSW"
#rcA7
robustness_checks['cthi_2021_rcA7'] = 1/2*(robustness_checks['cthi_2021_hs_rescaled'] + robustness_checks['log_cthi_2021_gsw_rescaled'])
robustness_checks['cthi_2021_def_rcA7'] = "HS and log(GSW) scaled to [0,10], then (HS + GSW)/2"

##B
robustness_checks = pd.merge(cthi_gsw_robust,robustness_checks,on="country_iso3",validate="1:1")

#rcB0--5
for i in range(5):
    var = f'cthi_gsw_B{i}'
    robustness_checks[var] = robustness_checks[var]/robustness_checks[var].sum()
    robustness_checks[f'cthi_2021_rcB{i}'] = (robustness_checks['cthi_2021_hs']**3 * robustness_checks[var]**(1/3))/100
robustness_checks['cthi_2021_def_rcB0'] = "Original CTHI"
robustness_checks['cthi_2021_def_rcB1'] = "Only use inward FDI"
robustness_checks['cthi_2021_def_rcB2'] = "Only use outward FDI"
robustness_checks['cthi_2021_def_rcB3'] = "Don't adjust at bilateral level and take only reported"
robustness_checks['cthi_2021_def_rcB4'] = "Don't adjust at bilateral level and take only derived"

##C
robustness_checks['cthi_2021_rcC0'] = robustness_checks['cthi_2021_rcA0']
robustness_checks['cthi_2021_def_rcC0'] = "Original CTHI"
#rcC1
hi_cols = [f"cthi_2021_hs_hi{i}" for i in range(1,21)]
robustness_checks["cthi_2021_hs_C1"] =  robustness_checks[hi_cols].mean(1)
robustness_checks['cthi_2021_rcC1'] = (robustness_checks['cthi_2021_hs_C1']**3 * robustness_checks['cthi_2021_gsw']**(1/3))/100
robustness_checks['cthi_2021_def_rcC1'] = "Take arithmetic mean of 20 haven indicators to calculate HS"
#rcC2
hi_cat_cols = [f"cthi_2021_hs_cat{i}" for i in range(1,6)]
robustness_checks["cthi_2021_hs_C2"] =  robustness_checks[hi_cat_cols].apply(gmean, axis=1)
robustness_checks['cthi_2021_rcC2'] = (robustness_checks['cthi_2021_hs_C2']**3 * robustness_checks['cthi_2021_gsw']**(1/3))/100
robustness_checks['cthi_2021_def_rcC2'] = "Take geometric mean of 5 categories to calculate HS"
#rcC2-7
for i in range(1,6):
    hi_cat_cols = [f"cthi_2021_hs_cat{ind}" for ind in range(1,6) if ind != i]
    robustness_checks[f"cthi_2021_hs_C{i+2}"] =  robustness_checks[hi_cat_cols].mean(1)
    robustness_checks[f'cthi_2021_rcC{i+2}'] = (robustness_checks[f"cthi_2021_hs_C{i+2}"]**3 * robustness_checks['cthi_2021_gsw']**(1/3))/100
    robustness_checks[f'cthi_2021_def_rcC{i+2}'] = f"Discard Category {i}"

versions = [f"A{i}" for i in range(8)] + [f"B{i}" for i in range(5)] + [f"C{i}" for i in range(8)]
for version in versions:
    robustness_checks['cthi_2021_rank_rc'+str(version)] = robustness_checks['cthi_2021_rc'+str(version)].rank(method='min',ascending=False)
    robustness_checks['cthi_2021_share_rc'+str(version)] = robustness_checks['cthi_2021_rc'+str(version)] / robustness_checks['cthi_2021_rc'+str(version)].sum()

NameError: name 'cthi' is not defined

In [ ]:
import seaborn as snd
import pylab as plt
%matplotlib inline

for cat in ["A","B","C"]:
    corr = robustness_checks[[_ for _ in robustness_checks.columns if f"rank_rc{cat}" in _]]
    corr.columns = [_[-2:] for _ in corr.columns]
    # Compute the correlation matrix
    corr = corr.corr(method="spearman")

    # Generate a mask for the upper triangle
    mask = np.triu(np.ones_like(corr, dtype=bool))

    # Set up the matplotlib figure
    f, ax = plt.subplots(figsize=(5, 5))

    # Draw the heatmap with the mask and correct aspect ratio
    sns.heatmap(corr, mask=mask, cmap="Blues", vmax=1.,vmin=0,
                square=True, linewidths=.5, cbar_kws={"shrink": .5},annot=True,fmt="1.2f")
    plt.yticks(rotation=0)
    plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Robustness checks/checks_{cat}.svg')
    plt.show()

In [ ]:
##Correlation with harm
#SOTJ 202 data
sotj = pd.read_excel(f"{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Scale of Tax Injustice/State of Tax Justice report/combined_output.xlsx",engine="openpyxl")
sotj = sotj[["ISO-3 code of country","TA: Tax base gain (USD million)","TA: Tax base loss (USD million)"]]
sotj["Tax_avoidance_harm"] = sotj["TA: Tax base gain (USD million)"] - sotj["TA: Tax base loss (USD million)"]
sotj["ISO3"] = sotj["ISO-3 code of country"]
sotj =  sotj[["ISO3","Tax_avoidance_harm"]]
robustness_sotj = pd.merge(robustness_checks,sotj,left_on="country_iso3",right_on=["ISO3"])
robustness_sotj = robustness_sotj.loc[robustness_sotj["Tax_avoidance_harm"]>0]
len(robustness_sotj)

In [ ]:
for cat in ["A","B","C"]:
    corr = robustness_sotj[[_ for _ in robustness_sotj.columns if (f"share_rc{cat}" in _) or _ =="Tax_avoidance_harm"]]
    corr.columns = [_[-2:] if _ != "Tax_avoidance_harm" else "SOTJ" for _ in corr.columns ]
    # Compute the correlation matrix
    corr = corr.corr(method="spearman")

    # Generate a mask for the upper triangle
    mask = np.triu(np.ones_like(corr, dtype=bool))

    # Set up the matplotlib figure
    f, ax = plt.subplots(figsize=(5, 5))

    # Draw the heatmap with the mask and correct aspect ratio
    sns.heatmap(corr, mask=mask, cmap="Blues", vmax=1.,vmin=0,
                square=True, linewidths=.5, cbar_kws={"shrink": .5},annot=True,fmt="1.2f")
    plt.yticks(rotation=0)
    plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Robustness checks/checks_sotj_{cat}.svg')
    plt.show()

In [ ]:
corr = robustness_sotj[[_ for _ in robustness_sotj.columns if (f"share_rcA" in _) or (f"share_rcB" in _) or(f"share_rcC" in _) or _ =="Tax_avoidance_harm"]]
corr.columns = [_[-2:] if _ != "Tax_avoidance_harm" else "SOTJ" for _ in corr.columns ]
    # Compute the correlation matrix
corr = corr.corr(method="pearson")

    # Generate a mask for the upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

    # Set up the matplotlib figure
f, ax = plt.subplots(figsize=(15, 15))

    # Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, mask=mask, cmap="Blues", vmax=1.,vmin=0,
                square=True, linewidths=.5, cbar_kws={"shrink": .5},annot=True,fmt="1.1f")
plt.yticks(rotation=0)

plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Robustness checks/checks_sotj_all.svg')
plt.show()

In [ ]:
##EXCEL 2 - ROBUSTNESS CHECKS FILE

versions = [f"A{i}" for i in range(8)] + [f"B{i}" for i in range(5)] + [f"C{i}" for i in range(8)]
cols = []
cols_def = []
for version in versions:
    cols.append(f"cthi_2021_rc{version}")
    cols.append(f"cthi_2021_rank_rc{version}")
    cols_def.append(f"cthi_2021_def_rc{version}")


writer = pd.ExcelWriter(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/210308_file2_cthi_2021_robustnessChecks.xlsx', engine='xlsxwriter')
workbook  = writer.book
cell_format_num2_leftborder = workbook.add_format({'align':'center','num_format':'#,##0.0','left':True})

#Sheet 1: Alternative CTHI2021 results
robustness_checks_toexport = robustness_checks[['country','country_iso2','country_iso3',"income_final_2018","region_final","EU-28","EU-28 OCTs","EU-27","EU-27 OCTs","UK OCTs","OECD","OECD OCTs",'included_in_cthi_2021','included_in_cthi_2019'] + cols]
robustness_checks_toexport_sorted = robustness_checks_toexport.sort_values(by=['cthi_2021_rcA0'], ascending=False)
robustness_checks_toexport_sorted.to_excel(writer, sheet_name='Alternative CTHI2021 results', index=False)
worksheet = writer.sheets['Alternative CTHI2021 results']
worksheet.set_column('A:A', 26, None)
worksheet.set_column('B:N', None, None, {'hidden':True})
worksheet.set_column('O:BD', 15, None)
worksheet.freeze_panes(1, 1)
columns = ['Q','S','U','W','Y','AA','AC','AE','AG','AI','AK','AM','AO','AQ','AS','AU','AW','AY','BA','BC']
for column in columns:
    worksheet.set_column(column+':'+column, None, cell_format_num2_leftborder)

#Sheet 2: Alternative CTHI2021 results
#robustness_checks_toreshape = robustness_checks[['country','cthi_2021_def_rcA0','cthi_2021_def_rcA1','cthi_2021_def_rcA2','cthi_2021_def_rcA3','cthi_2021_def_rcA4','cthi_2021_def_rcA5','cthi_2021_def_rcA6','cthi_2021_def_rcA7','cthi_2021_def_rcB0','cthi_2021_def_rcB1','cthi_2021_def_rcB2','cthi_2021_def_rcB3','cthi_2021_def_rcB4','cthi_2021_def_rcC0','cthi_2021_def_rcC1','cthi_2021_def_rcC2','cthi_2021_def_rcC3','cthi_2021_def_rcC4','cthi_2021_def_rcC5','cthi_2021_def_rcC6','cthi_2021_def_rcC7']]
robustness_checks_toreshape = robustness_checks[['country'] + cols_def]
robustness_checks_toreshape = robustness_checks_toreshape[robustness_checks_toreshape['country'] == "Andorra"]
robustness_checks_reshaped = pd.melt(robustness_checks_toreshape.reset_index(), id_vars=['country'], value_vars=['cthi_2021_def_rcA0','cthi_2021_def_rcA1','cthi_2021_def_rcA2','cthi_2021_def_rcA3','cthi_2021_def_rcA4','cthi_2021_def_rcA5','cthi_2021_def_rcA6','cthi_2021_def_rcA7','cthi_2021_def_rcB0','cthi_2021_def_rcB1','cthi_2021_def_rcB2','cthi_2021_def_rcB3','cthi_2021_def_rcB4','cthi_2021_def_rcC0','cthi_2021_def_rcC1','cthi_2021_def_rcC2','cthi_2021_def_rcC3','cthi_2021_def_rcC4','cthi_2021_def_rcC5','cthi_2021_def_rcC6','cthi_2021_def_rcC7'], var_name='Version of robustness check', value_name='Definition')
#robustness_checks_reshaped = pd.melt(robustness_checks_toreshape.reset_index(), id_vars=['country'], value_vars=[['country'] + cols_def], var_name='Version of robustness check', value_name='Definition')
robustness_checks_reshaped = robustness_checks_reshaped.drop('country', axis=1)
robustness_checks_reshaped.to_excel(writer, sheet_name='Definition of versions', index=False)
worksheet = writer.sheets['Definition of versions']
worksheet.set_column('A:B', 25, None)

writer.save()
writer.close()

In [ ]:
##EXCEL 3 - PUBLIC RESULTS FILE WITH FORMULAS AND NICE FORMATTING
writer = pd.ExcelWriter(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/210308_file3_cthi_2021_publicFullResults.xlsx', engine='xlsxwriter')
workbook  = writer.book

number_of_cthi_2021_jurisdictions = 70

cthi_2021_short = cthi[['cthi_2021_rank','country','country_iso2','country_iso3','cthi_2021','cthi_2021_gsw','cthi_2021_hs','included_in_cthi_2021']]
cthi_2021_short_sorted = cthi_2021_short.sort_values(by=['cthi_2021'], ascending=False)
cthi_2021_short_sorted.rename(columns = {'cthi_2021_rank':'Rank','country':'Country','country_iso2':'ISO-2','country_iso3':'ISO-3','cthi_2021_gsw':'Global Scale Weight (%)', 'cthi_2021_hs':'Haven Score'}, inplace = True)
cthi_2021_short_sorted = cthi_2021_short_sorted[(cthi_2021_short_sorted['included_in_cthi_2021']==1)]
del cthi_2021_short_sorted['cthi_2021']
del cthi_2021_short_sorted['included_in_cthi_2021']
cthi_2021_short_sorted.to_excel(writer, sheet_name='CTHI 2021 results', index=False)
worksheet = writer.sheets['CTHI 2021 results']

cell_format_header = workbook.add_format({'bold': True, 'align': 'center'})
cell_format_header.set_right()
cell_format_header.set_bottom()

#Add formula for CTHI value
for row in range(2,number_of_cthi_2021_jurisdictions+2):
    formula = f'=(E{row}^(1/3)*F{row}^(3))/100'
    worksheet.write_formula(f"G{row}", formula)
worksheet.write('G1', 'CTHI value', cell_format_header)

#Add formula for CTHI share (contribution)
for row in range(2,number_of_cthi_2021_jurisdictions+2):
    formula = f'=(G{row}/SUM(G:G))'
    worksheet.write_formula(f"H{row}", formula)
worksheet.write('H1', 'CTHI share (%)', cell_format_header)

#Format columns
cell_format_pct = workbook.add_format({'align':'center'})
cell_format_pct.set_num_format(10)
cell_format_num2 = workbook.add_format({'align':'center','num_format': '#,###.0'})
cell_format_center = workbook.add_format({'align':'center'})
cell_format_left = workbook.add_format({'align':'left'})
cell_format_left_header = workbook.add_format({'align':'left','bold': True})
cell_format_left_header.set_right()
cell_format_left_header.set_bottom()
worksheet.set_column('A:A', 6, cell_format_center)
worksheet.set_column('B:B', 26, cell_format_left)
worksheet.write(0, 1, 'Country', cell_format_left_header)
worksheet.set_column('C:C', 6, cell_format_center)
worksheet.set_column('D:D', 6, cell_format_center)
worksheet.set_column('E:E', 22, cell_format_pct)
worksheet.set_column('F:F', 15, cell_format_num2)
worksheet.set_column('G:G', 15, cell_format_num2)
worksheet.set_column('H:H', 15, cell_format_pct)

#Conditional formatting
format_color1 = workbook.add_format({'bg_color':'#6BACD1'})
format_color2 = workbook.add_format({'bg_color':'#FEED7B'})
format_color3 = workbook.add_format({'bg_color':'#FFA64D'})
format_color4 = workbook.add_format({'bg_color':'#C94741'})
worksheet.conditional_format('F2:F71', {'type': 'cell', 'criteria': 'between', 'minimum': 0, 'maximum': 24.99, 'format': format_color1})
worksheet.conditional_format('F2:F71', {'type': 'cell', 'criteria': 'between', 'minimum': 25, 'maximum': 49.99, 'format': format_color2})
worksheet.conditional_format('F2:F71', {'type': 'cell', 'criteria': 'between', 'minimum': 50, 'maximum': 74.99, 'format': format_color3})
worksheet.conditional_format('F2:F71', {'type': 'cell', 'criteria': 'between', 'minimum': 75, 'maximum': 101, 'format': format_color4})

#Endnote
worksheet.write('A73', 'The full results and methodology of the Corporate Tax Haven Index can be found at:', cell_format_left)
worksheet.write_url('A74', 'http://cthi.taxjustice.net/')

writer.save()

In [ ]:
##EXCEL 4 - Explanations (export only once, then add explanations manually)
#writer = pd.ExcelWriter('C:/Users/miros/Tax Justice Network Ltd/TJN - Shared Documents (1)/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/210202_file4_cthi_2021_changesExplanations_HSfrom210124.xlsx', engine='xlsxwriter')
writer = pd.ExcelWriter('C:/Users/miros/Desktop/Dropbox/Research/2005 CTHI 2021/210304_file4_cthi_2021_comms_toEdit.xlsx', engine='xlsxwriter') #TODO find this file
workbook  = writer.book

#Sheet 1: Explanations of changes
cthi['Changes: methodology'] = ""
cthi['Changes: legislation'] = ""
cthi['Changes: own'] = ""
data_explanations = cthi[['country','Changes: methodology','Changes: legislation','country_iso2','country_iso3',"income_final_2018","region_final","EU-28","EU-28 OCTs","EU-27","EU-27 OCTs","UK OCTs","OECD","OECD OCTs",'included_in_cthi_2021','included_in_cthi_2019','cthi_2021','cthi_2021_rank','cthi_2019','cthi_2019_rank','cthi_2021_rank - cthi_2019_rank','CTHI: old GSW, old HS, old countries','CTHI rank: old GSW, old HS, old countries','CTHI: new GSW, old HS, old countries','CTHI rank: new GSW, old HS, old countries','CTHI: new GSW, new HS, old countries','CTHI rank: new GSW, new HS, old countries','CTHI: new GSW, new HS, new countries','CTHI rank: new GSW, new HS, new countries','Total change in rank','Rank change due to GSW','Rank change due to HS','Rank change due to new countries','included_in_cthi_2021','included_in_cthi_2019','cthi_2021 - cthi_2019','cthi_2021 - cthi_2019 (%)','cthi_2021_rank','cthi_2019_rank','cthi_2021_rank - cthi_2019_rank','cthi_2021_rank - cthi_2019_rank (%)','cthi_2021_share','cthi_2019_share','cthi_2021_share - cthi_2019_share','cthi_2021_share - cthi_2019_share (%)','cthi_2021_gsw','cthi_2019_gsw','cthi_2021_gsw - cthi_2019_gsw','cthi_2021_gsw - cthi_2019_gsw (%)', 'cthi_2021_hs','cthi_2019_hs','cthi_2021_hs - cthi_2019_hs','cthi_2021_hs - cthi_2019_hs (%)', 'cthi_2021_hs_cat1','cthi_2019_hs_cat1','cthi_2021_hs_cat1 - cthi_2019_hs_cat1','cthi_2021_hs_cat1 - cthi_2019_hs_cat1 (%)', 'cthi_2021_hs_cat2','cthi_2019_hs_cat2','cthi_2021_hs_cat2 - cthi_2019_hs_cat2','cthi_2021_hs_cat2 - cthi_2019_hs_cat2 (%)', 'cthi_2021_hs_cat3','cthi_2019_hs_cat3','cthi_2021_hs_cat3 - cthi_2019_hs_cat3','cthi_2021_hs_cat3 - cthi_2019_hs_cat3 (%)', 'cthi_2021_hs_cat4','cthi_2019_hs_cat4','cthi_2021_hs_cat4 - cthi_2019_hs_cat4','cthi_2021_hs_cat4 - cthi_2019_hs_cat4 (%)', 'cthi_2021_hs_cat5','cthi_2019_hs_cat5','cthi_2021_hs_cat5 - cthi_2019_hs_cat5','cthi_2021_hs_cat5 - cthi_2019_hs_cat5 (%)', 'cthi_2021_hs_hi1','cthi_2019_hs_hi1','cthi_2021_hs_hi1 - cthi_2019_hs_hi1','cthi_2021_hs_hi1 - cthi_2019_hs_hi1 (%)', 'cthi_2021_hs_hi2','cthi_2019_hs_hi2','cthi_2021_hs_hi2 - cthi_2019_hs_hi2','cthi_2021_hs_hi2 - cthi_2019_hs_hi2 (%)', 'cthi_2021_hs_hi3','cthi_2019_hs_hi3','cthi_2021_hs_hi3 - cthi_2019_hs_hi3','cthi_2021_hs_hi3 - cthi_2019_hs_hi3 (%)', 'cthi_2021_hs_hi4','cthi_2019_hs_hi4','cthi_2021_hs_hi4 - cthi_2019_hs_hi4','cthi_2021_hs_hi4 - cthi_2019_hs_hi4 (%)', 'cthi_2021_hs_hi5','cthi_2019_hs_hi5','cthi_2021_hs_hi5 - cthi_2019_hs_hi5','cthi_2021_hs_hi5 - cthi_2019_hs_hi5 (%)', 'cthi_2021_hs_hi6','cthi_2019_hs_hi6','cthi_2021_hs_hi6 - cthi_2019_hs_hi6','cthi_2021_hs_hi6 - cthi_2019_hs_hi6 (%)', 'cthi_2021_hs_hi7','cthi_2019_hs_hi7','cthi_2021_hs_hi7 - cthi_2019_hs_hi7','cthi_2021_hs_hi7 - cthi_2019_hs_hi7 (%)', 'cthi_2021_hs_hi8','cthi_2019_hs_hi8','cthi_2021_hs_hi8 - cthi_2019_hs_hi8','cthi_2021_hs_hi8 - cthi_2019_hs_hi8 (%)', 'cthi_2021_hs_hi9','cthi_2019_hs_hi9','cthi_2021_hs_hi9 - cthi_2019_hs_hi9','cthi_2021_hs_hi9 - cthi_2019_hs_hi9 (%)', 'cthi_2021_hs_hi10','cthi_2019_hs_hi10','cthi_2021_hs_hi10 - cthi_2019_hs_hi10','cthi_2021_hs_hi10 - cthi_2019_hs_hi10 (%)', 'cthi_2021_hs_hi11','cthi_2019_hs_hi11','cthi_2021_hs_hi11 - cthi_2019_hs_hi11','cthi_2021_hs_hi11 - cthi_2019_hs_hi11 (%)', 'cthi_2021_hs_hi12','cthi_2019_hs_hi12','cthi_2021_hs_hi12 - cthi_2019_hs_hi12','cthi_2021_hs_hi12 - cthi_2019_hs_hi12 (%)', 'cthi_2021_hs_hi13','cthi_2019_hs_hi13','cthi_2021_hs_hi13 - cthi_2019_hs_hi13','cthi_2021_hs_hi13 - cthi_2019_hs_hi13 (%)', 'cthi_2021_hs_hi14','cthi_2019_hs_hi14','cthi_2021_hs_hi14 - cthi_2019_hs_hi14','cthi_2021_hs_hi14 - cthi_2019_hs_hi14 (%)', 'cthi_2021_hs_hi15','cthi_2019_hs_hi15','cthi_2021_hs_hi15 - cthi_2019_hs_hi15','cthi_2021_hs_hi15 - cthi_2019_hs_hi15 (%)', 'cthi_2021_hs_hi16','cthi_2019_hs_hi16','cthi_2021_hs_hi16 - cthi_2019_hs_hi16','cthi_2021_hs_hi16 - cthi_2019_hs_hi16 (%)', 'cthi_2021_hs_hi17','cthi_2019_hs_hi17','cthi_2021_hs_hi17 - cthi_2019_hs_hi17','cthi_2021_hs_hi17 - cthi_2019_hs_hi17 (%)', 'cthi_2021_hs_hi18','cthi_2019_hs_hi18','cthi_2021_hs_hi18 - cthi_2019_hs_hi18','cthi_2021_hs_hi18 - cthi_2019_hs_hi18 (%)', 'cthi_2021_hs_hi19','cthi_2019_hs_hi19','cthi_2021_hs_hi19 - cthi_2019_hs_hi19','cthi_2021_hs_hi19 - cthi_2019_hs_hi19 (%)', 'cthi_2021_hs_hi20','cthi_2019_hs_hi20','cthi_2021_hs_hi20 - cthi_2019_hs_hi20','cthi_2021_hs_hi20 - cthi_2019_hs_hi20 (%)']]
#Load explanations and merge them onto the dataset
cthi_2021_changesExplanations = pd.read_excel('C:/Users/miros/Desktop/Dropbox/Research/2005 CTHI 2021/CTHI 2021- MASTER score change explanation-LIVE-open online only-210205_1701.xlsx') #TODO find this file
cthi_2021_changesExplanations = cthi_2021_changesExplanations[cthi_2021_changesExplanations.Period == 21]
cthi_2021_changesExplanations = cthi_2021_changesExplanations[['Juris_Name','Explanation 1','Explanation 2','Explanation 3','Explanation 4','Explanation 5','Explanation 6','Explanation 7','Explanation 8','Explanation 9','Explanation 10','Explanation 11','Explanation 12','Explanation 13','Explanation 14','Explanation 15','Explanation 16','Explanation 17','Explanation 18','Explanation 19','Explanation 20']]
cthi_2021_changesExplanations.rename(columns = {'Juris_Name':'country'}, inplace = True)
data_explanations = pd.merge(data_explanations,cthi_2021_changesExplanations,on=['country'],how='outer')
#rank_changes_breakdown = rank_changes_breakdown.loc[~(rank_changes_breakdown['cthi_2021'].isna())]
data_explanations_sorted = data_explanations.sort_values(by=['cthi_2021'], ascending=False)
data_explanations_sorted = data_explanations_sorted.loc[~(data_explanations_sorted['cthi_2021'].isna())]
data_explanations_sorted.to_excel(writer, sheet_name='Comms', index=False)
worksheet = writer.sheets['Comms']
worksheet.set_column('A:A', 26, None)
worksheet.set_column('B:D', 20, None)
worksheet.freeze_panes(1, 1)
cell_format_fill = workbook.add_format({'bg_color':'#d4d4d4'})
worksheet.set_column('E:G', 20, cell_format_fill)

#Sheet 2: Cluster analysis
df_groups.to_excel(writer, sheet_name='Cluster analysis', index=False)
worksheet = writer.sheets['Cluster analysis']
worksheet.set_column('A:A', 26, None)
worksheet.freeze_panes(1, 3)

writer.save()

In [ ]:
df_groups_forCountryComparisons = df_groups.copy()
df_groups_forCountryComparisons['index'] = "1"
df_groups_forCountryComparisons_melted = df_groups_forCountryComparisons.melt(id_vars=["index", "Group"])


In [ ]:
##EXCEL 5 - Bilateral FDI
#writer = pd.ExcelWriter('C:/Users/miros/Tax Justice Network Ltd/TJN - Shared Documents (1)/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/210203_file5_cthi_2021_bilateral_HSfrom210202_toEdit.xlsx', engine='xlsxwriter')
writer = pd.ExcelWriter('C:/Users/miros/Desktop/Dropbox/Research/2005 CTHI 2021/210223_file5_cthi_2021_bilateral_HSfrom210211_toEdit.xlsx', engine='xlsxwriter') #TODO find this file
workbook  = writer.book

#Sheet 1: Bilateral FDI data
bilateral_fdi_reduced = bilateral_fdi.loc[~(bilateral_fdi['fdi_in_final'] == 0)]
bilateral_fdi_reduced = bilateral_fdi_reduced[['country','country_iso3','counterpart_country','counterpart_country_iso3','year','fdi_in_final','fdi_out_final','tot_cy_fdi_in_final','tot_cy_fdi_out_final']]
bilateral_fdi_reduced.rename(columns = {'tot_cy_fdi_in_final':'Inward FDI sum by country and year', 'tot_cy_fdi_out_final':'Outward FDI sum by country and year', 'fdi_in_final':'Inward FDI', 'fdi_out_final':'Outward FDI'}, inplace = True)
bilateral_fdi_reduced['Inward FDI: Rank of counterpart_country within country and year'] = bilateral_fdi_reduced.groupby(['country','year'])['Inward FDI'].transform('rank', method='min',ascending=False)
bilateral_fdi_reduced['Outward FDI: Rank of counterpart_country within country and year'] = bilateral_fdi_reduced.groupby(['country','year'])['Outward FDI'].transform('rank', method='min',ascending=False)

bilateral_fdi_reduced.to_excel(writer, sheet_name='Bilateral FDI', index=False)
worksheet = writer.sheets['Bilateral FDI']

writer.save()
writer.close()

In [ ]:
##EXCEL 6 - Partners and press
writer = pd.ExcelWriter(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/210308_file6_cthi_2021_partnersAndPress_toEdit.xlsx', engine='xlsxwriter')
#writer = pd.ExcelWriter('C:/Users/miros/Desktop/Dropbox/Research/2005 CTHI 2021/210226_file6_cthi_2021_partnersAndPress_HSfrom210223_toEdit.xlsx', engine='xlsxwriter')
workbook  = writer.book

number_of_cthi_2021_jurisdictions = 70

#Define formats
cell_format_num2 = workbook.add_format({'align':'center','num_format': '#,##0.0'})
cell_format_num3 = workbook.add_format({'align':'center','num_format': '#,##0'})
cell_format_pct_leftborder = workbook.add_format({'align':'center','num_format':10,'left':True})
cell_format_num2_leftborder = workbook.add_format({'align':'center','num_format':'#,##0.0','left':True})
cell_format_pct = workbook.add_format({'num_format':10})
cell_format_pct_rightborder = workbook.add_format({'align':'right','num_format':10,'right':True})
cell_format_rightborder = workbook.add_format({'right':True})
cell_format_left = workbook.add_format({'align' : 'left'})
cell_format_header = workbook.add_format({'bold': True, 'align': 'center'})
cell_format_header.set_right()
cell_format_header.set_bottom()
cell_format_wrap = workbook.add_format({'text_wrap': True})
cell_format_embargo = workbook.add_format({'bold': True, 'font_color': 'red','font_size' : '20'})
cell_format_left_header = workbook.add_format({'align':'left','bold': True})
cell_format_left_header.set_right()
cell_format_left_header.set_bottom()

#Sheet 1: CTHI2021 short
cthi_2021_short = cthi[['cthi_2021_rank','country','country_iso2','country_iso3','cthi_2021','cthi_2021_gsw','cthi_2021_hs','included_in_cthi_2021']]
cthi_2021_short_sorted = cthi_2021_short.sort_values(by=['cthi_2021'], ascending=False)
cthi_2021_short_sorted.rename(columns = {'cthi_2021_rank':'Rank','country':'Country','country_iso2':'ISO-2','country_iso3':'ISO-3','cthi_2021_gsw':'Global Scale Weight (%)', 'cthi_2021_hs':'Haven Score'}, inplace = True)
cthi_2021_short_sorted = cthi_2021_short_sorted[(cthi_2021_short_sorted['included_in_cthi_2021']==1)]
del cthi_2021_short_sorted['cthi_2021']
del cthi_2021_short_sorted['included_in_cthi_2021']
cthi_2021_short_sorted.to_excel(writer, sheet_name='CTHI 2021 results', index=False)
worksheet = writer.sheets['CTHI 2021 results']
#Add formula for CTHI value
for row in range(2,number_of_cthi_2021_jurisdictions+2):
    formula = f'=(E{row}^(1/3)*F{row}^(3))/100'
    worksheet.write_formula(f"G{row}", formula)
worksheet.write('G1', 'CTHI value', cell_format_header)
#Add formula for CTHI share (contribution)
for row in range(2,number_of_cthi_2021_jurisdictions+2):
    formula = f'=(G{row}/SUM(G:G))'
    worksheet.write_formula(f"H{row}", formula)
worksheet.write('H1', 'CTHI share (%)', cell_format_header)
#Format columns
cell_format_pct = workbook.add_format({'align':'center'})
cell_format_pct.set_num_format(10)
cell_format_num2 = workbook.add_format({'align':'center','num_format': '#,###.0'})
cell_format_center = workbook.add_format({'align':'center'})
cell_format_left = workbook.add_format({'align':'left'})
cell_format_left_header = workbook.add_format({'align':'left','bold': True})
cell_format_left_header.set_right()
cell_format_left_header.set_bottom()
worksheet.set_column('A:A', 6, None)
worksheet.write(0, 1, 'Country', cell_format_left_header)
worksheet.set_column('B:B', 26, cell_format_left)
worksheet.write(0, 1, 'Country', cell_format_left_header)
worksheet.set_column('C:C', 6, cell_format_center)
worksheet.set_column('D:D', 6, cell_format_center)
worksheet.set_column('E:E', 22, cell_format_pct)
worksheet.set_column('F:F', 15, cell_format_num2)
worksheet.set_column('G:G', 15, cell_format_num2)
worksheet.set_column('H:H', 15, cell_format_pct)
#Conditional formatting
format_color1 = workbook.add_format({'bg_color':'#6BACD1'})
format_color2 = workbook.add_format({'bg_color':'#FEED7B'})
format_color3 = workbook.add_format({'bg_color':'#FFA64D'})
format_color4 = workbook.add_format({'bg_color':'#C94741'})
worksheet.conditional_format('F2:F71', {'type': 'cell', 'criteria': 'between', 'minimum': 0, 'maximum': 24.99, 'format': format_color1})
worksheet.conditional_format('F2:F71', {'type': 'cell', 'criteria': 'between', 'minimum': 25, 'maximum': 49.99, 'format': format_color2})
worksheet.conditional_format('F2:F71', {'type': 'cell', 'criteria': 'between', 'minimum': 50, 'maximum': 74.99, 'format': format_color3})
worksheet.conditional_format('F2:F71', {'type': 'cell', 'criteria': 'between', 'minimum': 75, 'maximum': 101, 'format': format_color4})
#Endnotes
worksheet.write('A73', 'Once the embargo lifts, the full results and methodology of the Corporate Tax Haven Index can be found at:', cell_format_left)
worksheet.write_url('A74', 'http://cthi.taxjustice.net/') #maybe change to cthi.taxjustice.net if that will work by then?
worksheet.write('A76', 'All material in this Excel file is strictly embargoed for 00:01hrs GMT Tuesday 9 March 2021', cell_format_embargo)
worksheet.set_tab_color('#ed8f6d')

#Sheet 2: Haven Scores
cthi_2021_hs_toexport = cthi[['country','country_iso2','country_iso3','cthi_2021_hs_hi1','cthi_2021_hs_hi2','cthi_2021_hs_hi3','cthi_2021_hs_hi4','cthi_2021_hs_hi5','cthi_2021_hs_hi6','cthi_2021_hs_hi7','cthi_2021_hs_hi8','cthi_2021_hs_hi9','cthi_2021_hs_hi10','cthi_2021_hs_hi11','cthi_2021_hs_hi12','cthi_2021_hs_hi13','cthi_2021_hs_hi14','cthi_2021_hs_hi15','cthi_2021_hs_hi16','cthi_2021_hs_hi17','cthi_2021_hs_hi18','cthi_2021_hs_hi19','cthi_2021_hs_hi20','cthi_2021_hs_cat1','cthi_2021_hs_cat2','cthi_2021_hs_cat3','cthi_2021_hs_cat4','cthi_2021_hs_cat5','cthi_2021_hs']]
cthi_2021_hs_toexport.rename(columns = {"country" : "Country", "cthi_2021_hs_hi1" : "HI 1: Lowest available corporate income tax (LACIT)", "cthi_2021_hs_hi2" : "HI 2: Foreign investment income treatment", "cthi_2021_hs_hi3" : "HI 3: Loss utilisation", "cthi_2021_hs_hi4" : "HI 4: Capital gains taxation", "cthi_2021_hs_hi5" : "HI 5: Sectoral exemptions", "cthi_2021_hs_hi6" : "HI 6: Tax holidays and economic zones", "cthi_2021_hs_hi7" : "HI 7: Patent boxes", "cthi_2021_hs_hi8" : "HI 8: Fictional interest deduction", "cthi_2021_hs_hi9" : "HI 9: Public company accounts", "cthi_2021_hs_hi10" : "HI 10: Public country by country reporting", "cthi_2021_hs_hi11" : "HI 11: Local filing of country by country reports", "cthi_2021_hs_hi12" : "HI 12: Tax rulings and extractive contracts", "cthi_2021_hs_hi13" : "HI 13: Reporting of tax avoidance schemes", "cthi_2021_hs_hi14" : "HI 14: Tax court secrecy", "cthi_2021_hs_hi15" : "HI 15: Deduction limitation for interest", "cthi_2021_hs_hi16" : "HI 16: Deduction limitation for royalties", "cthi_2021_hs_hi17" : "HI 17: Deduction limitation for service payments", "cthi_2021_hs_hi18" : "HI 18: Dividend withholding tax", "cthi_2021_hs_hi19" : "HI 19: Controlled foreign company rules", "cthi_2021_hs_hi20" : "HI 20: Double tax treaty aggressiveness ", "cthi_2021_hs_cat1" : "Category 1: LACIT", "cthi_2021_hs_cat2" : "Category 2: Loopholes and gaps", "cthi_2021_hs_cat3" : "Category 3: Transparency", "cthi_2021_hs_cat4" : "Category 4: Anti-avoidance", "cthi_2021_hs_cat5" : "Category 5: Double tax treaty aggressiveness", 'cthi_2021_hs' : 'Haven Score'}, inplace = True)
cthi_2021_hs_toexport_sorted = cthi_2021_hs_toexport.sort_values(by=['Country'], ascending=True)
cthi_2021_hs_toexport_sorted = cthi_2021_hs_toexport_sorted[(cthi_2021_hs_toexport_sorted['Haven Score']>0)]
cthi_2021_hs_toexport_sorted.to_excel(writer, sheet_name='Haven Scores', index=False)
worksheet = writer.sheets['Haven Scores']
worksheet.set_column('A:A', 26, None)
worksheet.write(0, 0, 'Country', cell_format_left_header)
worksheet.set_column('B:C', None, None, {'hidden':True})
worksheet.set_column('D:AC', 15, None)
#worksheet.conditional_format('D1:AC1', {'type': 'no_errors','format': cell_format_header_wrap})
worksheet.freeze_panes(1, 1)
worksheet.set_tab_color('#ed8f6d')
#Add formula for Cat1
for row in range(2,number_of_cthi_2021_jurisdictions+2):
    formula = f'=(D{row})'
    worksheet.write_formula(f"X{row}", formula)
#Add formula for Cat2
for row in range(2,number_of_cthi_2021_jurisdictions+2):
    formula = f'=(E{row}+F{row}+G{row}+H{row}+I{row}+J{row}+K{row})/7'
    worksheet.write_formula(f"Y{row}", formula)
#Add formula for Cat3
for row in range(2,number_of_cthi_2021_jurisdictions+2):
    formula = f'=(L{row}+M{row}+N{row}+O{row}+P{row}+Q{row})/6'
    worksheet.write_formula(f"Z{row}", formula)
#Add formula for Cat4
for row in range(2,number_of_cthi_2021_jurisdictions+2):
    formula = f'=(R{row}+S{row}+T{row}+U{row}+V{row})/5'
    worksheet.write_formula(f"AA{row}", formula)
#Add formula for Cat5
for row in range(2,number_of_cthi_2021_jurisdictions+2):
    formula = f'=(W{row})'
    worksheet.write_formula(f"AB{row}", formula)
#Add formula for HS
for row in range(2,number_of_cthi_2021_jurisdictions+2):
    formula = f'=(X{row}+Y{row}+Z{row}+AA{row}+AB{row})/5'
    worksheet.write_formula(f"AC{row}", formula)
#Conditional formatting
worksheet.conditional_format('D2:AC71', {'type': 'cell', 'criteria': 'between', 'minimum': 0, 'maximum': 24.99, 'format': format_color1})
worksheet.conditional_format('D2:AC71', {'type': 'cell', 'criteria': 'between', 'minimum': 25, 'maximum': 49.99, 'format': format_color2})
worksheet.conditional_format('D2:AC71', {'type': 'cell', 'criteria': 'between', 'minimum': 50, 'maximum': 74.99, 'format': format_color3})
worksheet.conditional_format('D2:AC71', {'type': 'cell', 'criteria': 'between', 'minimum': 75, 'maximum': 101, 'format': format_color4})

#Sheet 3: Global Scale Weights
cthi_2021_gsw_toexport = cthi[['country','country_iso2','country_iso2','tot_cy_fdi_in_final_2019', 'tot_cy_fdi_out_final_2019','tot_y_fdi_final_2019','cthi_2021_gsw','included_in_cthi_2021']]
cthi_2021_gsw_toexport.rename(columns = {"country" : "Country", "tot_cy_fdi_in_final_2019" : "Inward FDI, 2019", "tot_cy_fdi_out_final_2019" : "Outward FDI, 2019", "tot_y_fdi_final_2019" : "Sum of inward and outward FDI, 2019", "cthi_2021_gsw" : "Global Scale Weight", 'included_in_cthi_2021' : "Included in CTHI 2021?"}, inplace = True)
cthi_2021_gsw_toexport = cthi_2021_gsw_toexport[(cthi_2021_gsw_toexport['Inward FDI, 2019'] != 0) & (cthi_2021_gsw_toexport['Outward FDI, 2019'] != 0)]
cthi_2021_gsw_toexport_sorted = cthi_2021_gsw_toexport.sort_values(by=['Global Scale Weight'], ascending=False)
cthi_2021_gsw_toexport_sorted.to_excel(writer, sheet_name='Global Scale Weights', index=False)
worksheet = writer.sheets['Global Scale Weights']
worksheet.set_column('A:A', 26, None)
worksheet.write(0, 0, 'Country', cell_format_left_header)
worksheet.set_column('B:C', None, None, {'hidden':True})
worksheet.set_column('D:E', 20, cell_format_num3)
worksheet.set_column('F:F', 25, cell_format_num3)
worksheet.set_column('G:G', 20, cell_format_pct)
worksheet.set_column('H:H', 25, None)
worksheet.freeze_panes(1, 1)
worksheet.set_tab_color('#ed8f6d')
#Add formula for sum of in+out
for row in range(2,248):
    formula = f'=(D{row}+E{row})'
    worksheet.write_formula(f"F{row}", formula)
#Add formula for GSW
for row in range(2,248):
    formula = f'=F{row}/SUM(F:F)'
    worksheet.write_formula(f"G{row}", formula)

#Sheet 4: Cluster analysis
df_groups.loc[(df_groups["Group"] == 'Higher income'), "Group"] = 'Higher income (1/2)'
df_groups.loc[(df_groups["Group"] == 'Lower income'), "Group"] = 'Lower income (2/2)'
df_groups.loc[(df_groups["Group"] == 'High income'), "Group"] = 'High income (1/4)'
df_groups.loc[(df_groups["Group"] == 'Low income'), "Group"] = 'Low income (2/4)'
df_groups.loc[(df_groups["Group"] == 'Lower middle income'), "Group"] = 'Lower middle income (3/4)'
df_groups.loc[(df_groups["Group"] == 'Upper middle income'), "Group"] = 'Upper middle income (4/4)'
df_groups.loc[(df_groups["Group"] == ''), "Group"] = 'Not rated'
df_groups.loc[(df_groups["Group"] == 'Not rated'), '"Region" or "Group"?'] = 'OECD transparency rating'
df_groups.loc[(df_groups["Group"] == 'Compliant'), '"Region" or "Group"?'] = 'OECD transparency rating'
df_groups.loc[(df_groups["Group"] == 'Largely Compliant'), '"Region" or "Group"?'] = 'OECD transparency rating'
df_groups.loc[(df_groups["Group"] == 'Non-Compliant'), '"Region" or "Group"?'] = 'OECD transparency rating'
df_groups.loc[(df_groups["Group"] == 'Partially Compliant'), '"Region" or "Group"?'] = 'OECD transparency rating'
df_groups.loc[(df_groups["Group"] == 'Provisionally Largely Compliant'), '"Region" or "Group"?'] = 'OECD transparency rating'
df_groups.loc[(df_groups["Group"] == 'Harmful'), '"Region" or "Group"?'] = 'OECD harmful regimes'
df_groups.loc[(df_groups["Group"] == 'Not harmful'), '"Region" or "Group"?'] = 'OECD harmful regimes'
df_groups.loc[(df_groups["Group"] == 'Under review'), '"Region" or "Group"?'] = 'OECD harmful regimes'
df_groups_reduced = df_groups[["Group",'"Region" or "Group"?',"CTHI 2021 share (sum)","GSW 2021 (sum)","HS 2021 (average)","HS 2021 category 1 (average)","HS 2021 category 2 (average)","HS 2021 category 3 (average)","HS 2021 category 4 (average)","HS 2021 category 5 (average)"]]
df_groups_reduced.to_excel(writer, sheet_name='Cluster analysis', index=False)
worksheet = writer.sheets['Cluster analysis']
worksheet.set_column('A:A', 35, None)
worksheet.write(0, 0, 'Country', cell_format_left_header)
worksheet.set_column('B:B', 15, None)
worksheet.set_column('C:D', 20, cell_format_pct)
worksheet.set_column('E:J', 20, cell_format_num2)
worksheet.freeze_panes(1, 2)
worksheet.set_tab_color('#f5da82')

#Sheet 5: Country characteristics
country_char = cthi[['country','country_iso2','country_iso3','All CTHI 2021 jurisdictions',"All CTHI 2019 jurisdictions","income_final_2018","IncomeClass2","region_final","EU-28","EU-28 OCTs","EU-28 + EU-28 OCTs","EU-27","EU-27 OCTs","EU-27 + EU-27 OCTs","UK","UK OCTs","UK + UK OCTs","OECD","OECD OCTs","OECD + OECD OCTs","UK","G20","Spider","Axis","Top 3","Top 5","Top 10","Top 15","EU blacklist (201006)","EU greylist (201006)","UNCTAD (2015) list of tax havens","trans_rating_oecd","harmful_regimes_oecd"]]
country_char.rename(columns = {'country':'Country','country_iso2':'ISO-2','country_iso3':'ISO-3'}, inplace = True)
country_char[["G20"]].astype(bool)
country_char_sorted = country_char.sort_values(by=['Country'], ascending=True)
country_char_sorted.to_excel(writer, sheet_name='Country characteristics', index=False)
worksheet = writer.sheets['Country characteristics']
worksheet.set_column('A:A', 26, None)
worksheet.write(0, 0, 'Country', cell_format_left_header)
worksheet.freeze_panes(1, 3)
worksheet.set_tab_color('#f5da82')

writer.save()
writer.close()

In [ ]:
##Master score change file
#Load old masterScoreChange file and the excel with new HS C:/Users\miros/Desktop/Dropbox/Research/2005 CTHI 2021
cthi_2021_masterScoreChange_old = pd.read_excel('C:/Users/miros/Tax Justice Network Ltd/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Verification/CTHI 2021- MASTER score change explanation-LIVE-open online only-210223_1756.xlsx')
#cthi_2021_masterScoreChange_old = pd.read_excel('C:/Users/miros/Desktop/Dropbox/Research/2005 CTHI 2021/CTHI 2021- MASTER score change explanation-LIVE-open online only.xlsx')
cthi_2021_hs_toMergeToMasterScoreChange = pd.read_excel('C:/Users/miros/Tax Justice Network Ltd/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/Exports from database/210223 Database export.xlsx')

cthi_2021_masterScoreChange_new = cthi_2021_masterScoreChange_old[['Helper','Change 1','Explanation 1','Change 2','Explanation 2','Change 3','Explanation 3','Change 4','Explanation 4','Change 5','Explanation 5','Change 6','Explanation 6','Change 7','Explanation 7','Change 8','Explanation 8','Change 9','Explanation 9','Change 10','Explanation 10','Change 11','Explanation 11','Change 12','Explanation 12','Change  13','Explanation 13','Change 14','Explanation 14','Change  15','Explanation 15','Change 16','Explanation 16','Change 17','Explanation 17','Change 18','Explanation 18','Change 19','Explanation 19','Change 20','Explanation 20']]

cthi_2021_masterScoreChange_new = pd.merge(cthi_2021_masterScoreChange_new,cthi_2021_hs_toMergeToMasterScoreChange,on=['Helper'],how='outer')

for i in range(1, 21):
    cthi_2021_masterScoreChange_new = cthi_2021_masterScoreChange_new.rename(columns={i:"HI "+str(i)})

cthi_2021_masterScoreChange_new = cthi_2021_masterScoreChange_new[['Helper','Jurisdiction','Juris_Name','Period','HI 1','Change 1','Explanation 1','HI 2','Change 2','Explanation 2','HI 3','Change 3','Explanation 3','HI 4','Change 4','Explanation 4','HI 5','Change 5','Explanation 5','HI 6','Change 6','Explanation 6','HI 7','Change 7','Explanation 7','HI 8','Change 8','Explanation 8','HI 9','Change 9','Explanation 9','HI 10','Change 10','Explanation 10','HI 11','Change 11','Explanation 11','HI 12','Change 12','Explanation 12','HI 13','Change  13','Explanation 13','HI 14','Change 14','Explanation 14','HI 15','Change  15','Explanation 15','HI 16','Change 16','Explanation 16','HI 17','Change 17','Explanation 17','HI 18','Change 18','Explanation 18','HI 19','Change 19','Explanation 19','HI 20','Change 20','Explanation 20']]

#Export to new masterScoreChange excel file
writer = pd.ExcelWriter(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Verification/CTHI 2021- MASTER score change explanation-LIVE-open online only-210223_2345.xlsx', engine='xlsxwriter')
cthi_2021_masterScoreChange_new.to_excel(writer, sheet_name='Sheet1', index=False)
workbook  = writer.book
worksheet = writer.sheets['Sheet1']

#Formulas to calculate the change
rows = [3,5,7,9,11,13,15,17,19,21,23,25,27,29,31,33,35,37,39,41,43,45,47,49,51,53,55,57,59,61,63,65,67,69,71,73,75,77,79,81,83,85,87,89,91,93,95,97,99,101,103,105,107,109,111,113,115,117,119,121,123,125,127,129,131,133,135]
for row in rows:
    formula = f'=(E{row}-E{row-1})'
    worksheet.write_formula(f"F{row}", formula)
    formula = f'=(H{row}-H{row-1})'
    worksheet.write_formula(f"I{row}", formula)
    formula = f'=(K{row}-K{row-1})'
    worksheet.write_formula(f"L{row}", formula)
    formula = f'=(N{row}-N{row-1})'
    worksheet.write_formula(f"O{row}", formula)
    formula = f'=(Q{row}-Q{row-1})'
    worksheet.write_formula(f"R{row}", formula)
    formula = f'=(T{row}-T{row-1})'
    worksheet.write_formula(f"U{row}", formula)
    formula = f'=(W{row}-W{row-1})'
    worksheet.write_formula(f"X{row}", formula)
    formula = f'=(Z{row}-Z{row-1})'
    worksheet.write_formula(f"AA{row}", formula)
    formula = f'=(AC{row}-AC{row-1})'
    worksheet.write_formula(f"AD{row}", formula)
    formula = f'=(AF{row}-AF{row-1})'
    worksheet.write_formula(f"AG{row}", formula)
    formula = f'=(AI{row}-AI{row-1})'
    worksheet.write_formula(f"AJ{row}", formula)
    formula = f'=(AL{row}-AL{row-1})'
    worksheet.write_formula(f"AM{row}", formula)
    formula = f'=(AO{row}-AO{row-1})'
    worksheet.write_formula(f"AP{row}", formula)
    formula = f'=(AR{row}-AR{row-1})'
    worksheet.write_formula(f"AS{row}", formula)
    formula = f'=(AU{row}-AU{row-1})'
    worksheet.write_formula(f"AV{row}", formula)
    formula = f'=(AX{row}-AX{row-1})'
    worksheet.write_formula(f"AY{row}", formula)
    formula = f'=(BA{row}-BA{row-1})'
    worksheet.write_formula(f"BB{row}", formula)
    formula = f'=(BD{row}-BD{row-1})'
    worksheet.write_formula(f"BE{row}", formula)
    formula = f'=(BG{row}-BG{row-1})'
    worksheet.write_formula(f"BH{row}", formula)
    formula = f'=(BJ{row}-BJ{row-1})'
    worksheet.write_formula(f"BK{row}", formula)

#Conditional formatting
columns_changes = ['F2:F129','I2:I129','L2:L129','O2:O129','R2:R129','U2:U129','X2:X129','AA2:AA129','AD2:AD129','AG2:AG129','AJ2:AJ129','AM2:AM129','AP2:AP129','AS2:AS129','AV2:AV129','AY2:AY129','BB2:BB129','BE2:BE129','BH2:BH129','BK2:BK129']
format_color1 = workbook.add_format({'bg_color':'#6BACD1'})
format_color4 = workbook.add_format({'bg_color':'#C94741'})
for column in columns_changes:
    worksheet.conditional_format(column, {'type': 'cell', 'criteria': 'between', 'minimum': 0.0001, 'maximum': 150, 'format': format_color4})
    worksheet.conditional_format(column, {'type': 'cell', 'criteria': 'between', 'minimum': -0.0001, 'maximum': -150, 'format': format_color1})

worksheet.freeze_panes(1, 4)
writer.save()
writer.close()

Figures

In [ ]:
#GSW over time (2009-2019), individual country
country = 'DEU'
df = cthi_gsw_historical.loc[cthi_gsw_historical['country_iso3'] == country]
maxyrange = df['cthi_gsw'].max()
plt.plot(df['year'], df['cthi_gsw'])
plt.ylabel('Global Scale Weigth')
plt.axis([2008, 2020, 0, maxyrange+0.5*maxyrange])
plt.legend([country])


In [ ]:
plt.style.use('default')

In [ ]:
#Histograms
df = cthi.copy()
df = df[~(df['included_in_cthi_2021']==0)]
for i in range(1,6):
    plt.figure()
    plt.hist(df['cthi_2021_hs_cat'+str(i)], 100)
    plt.title('Histogram of CTHI 2021 HS cat '+str(i))
    plt.ylabel('Number of jurisdictions')
    plt.xlabel('Score')
    plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/Figures/Histograms/hist_cthi_2021_hs_cat_'+str(i)+'.svg')
    plt.close()

plt.figure()
plt.hist(df['cthi_2021_hs'], 25,color = "#564A5C")
#plt.title('Histogram of CTHI 2021 HS')
plt.ylabel('Number of jurisdictions')
plt.xlabel('Haven Score')
plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/Figures/Histograms/hist_cthi_2021_hs.svg')
plt.close()

plt.figure()
plt.hist(df['cthi_2021_gsw'], 25,color = "#564A5C")
#plt.title('Histogram of CTHI 2021 GSW')
plt.ylabel('Number of jurisdictions')
plt.xlabel('Global Scale Weight')
plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/Figures/Histograms/hist_cthi_2021_gsw.svg')
plt.close()

In [ ]:
df = cthi.copy()
df = df[~(df['included_in_cthi_2019']==0)]
for i in range(1,2):
    plt.figure()
    df = df.sort_values('cthi_2021_hs_hi'+str(i)+' - cthi_2019_hs_hi'+str(i))
    plt.barh(df['country'],df['cthi_2021_hs_hi'+str(i)+' - cthi_2019_hs_hi'+str(i)])
    plt.tight_layout()
    ax = fig.add_axes([0.1, 0.1, 0.8, 0.8]) # main axes
    ax.set_xticks([-100,-80,-60,-40,-20,0,20,40,60,80,100])
    plt.title('Change in Haven Indicator '+str(i)+', 2021 minus 2019')
    plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents (1)/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/Figures/Change_2021_2019/change_cthi_2021_2019_hs_hi'+str(i))
    plt.close()

In [ ]:
##Swarm plot for each country's HS
df = cthi.copy()
country = "Germany"
df["country_of_interest"] = country
sns.set_theme(style="whitegrid")
ax = sns.swarmplot(y="cthi_2021_hs", data=df, hue="Region", palette="Set2", dodge=True)

In [ ]:
##Swarm plot for each country's GSW
df = cthi.copy()
country = "Germany"
df["country_of_interest"] = country
sns.set_theme(style="whitegrid")
ax = sns.swarmplot(y="cthi_2021_gsw", data=df, hue="Region", palette="Set2", dodge=True)

In [ ]:
import matplotlib
##Treemap for CTHI share
df = cthi[['country_iso3','cthi_2021_share','included_in_cthi_2021','cthi_2021_hs','OECD',"OECD + OECD OCTs","th_eu_blacklist_201006",'th_eu_greylist_201006']]
df = df.sort_values(by=['cthi_2021_share'], ascending=False)
df = df[~(df['included_in_cthi_2021']==0)]
#Set sizes and color code key
sizes = df['cthi_2021_share'].copy()
color_code = df["cthi_2021_hs"].copy()
#Set colors
cmap = matplotlib.cm.Blues
mini=min(color_code)
maxi=max(color_code)
norm = matplotlib.colors.Normalize(vmin=mini, vmax=maxi)
colors = [cmap(norm(value)) for value in color_code]
#Set labels
labels = df['country_iso3'].copy()
labels[sizes<0.002] = "" 
#Plot and save
fig = plt.figure(figsize=(7, 5))
squarify.plot(sizes=sizes, label=labels, alpha=0.8, color=colors)
plt.axis('off')
plt.tight_layout()
plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/Figures/treemap.png',bbox_inches="tight",dpi=300)
#plt.show()
plt.gcf()

#plt.savefig('C:/Users/miros/Tax Justice Network Ltd/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/Figures/treemap.png')
#plt.show()

In [ ]:
#Plot treemap
plt.figure(figsize=(8,4))
sizes = df_flow[flow]/df_flow[flow].sum()
                #Label with country name + flow size
                if ("Trade" in flow) or ("Export" in flow) or ("Import" in flow) or ("FDI" in flow):
                    asterisk = [False]*len(df_flow)#df_flow["CbCR_OECD_Info_in"].fillna(False) == False
                elif ("PortI" in flow) or ("Banking" in flow):
                    asterisk = df_flow["AIE_OECD_Info_in"].fillna(False) == False
                else:
                    raise("error in columns")
                
                label = df_flow["p_name"].copy()
                label[asterisk] += "*"
                label += "\n"+df_flow[flow].map(normalize_number)
                label2 = df_flow["p_name"].map(get_iso3).values
                label2[asterisk] += "*"
                #Color with secrecy score
                color = df_flow[norm].map(get_secrecy_color).values
                #Keep only iso codes if the square is small
                label[sizes<0.01] = label2[sizes<0.01]
                #Don't label if the square is very small
                label[sizes<0.003] = "" 
                squarify.plot(sizes=sizes, label=label, color=color, alpha=1 )
                plt.axis('off')
                plt.tight_layout()
                print("{}/Tables/Country_vulnerabilities/{}/_agg_{}.png".format(version,country,flow))
                plt.savefig("{}/Tables/Country_vulnerabilities/{}/_agg_{}.png".format(version,country,flow),
                            bbox_inches="tight",dpi=300)
                plt.show()
                plt.gcf()

In [ ]:
#Dot chart comparing LACIT and Statutory rate (for Lucas)
df = pd.read_csv('C:/Users/miros/Desktop/Dropbox/Research/2005 CTHI 2021/lacit_statutory.csv')
# Reorder it following the values of the first value:
ordered_df = df.sort_values(by='Statutory')
my_range=range(1,len(df.index)+1)
# The horizontal plot is made using the hline function
plt.figure(figsize=(6,16))
plt.subplots_adjust(bottom=.25, left=.4)
plt.hlines(y=my_range, xmin=ordered_df['Statutory'], xmax=ordered_df['LACIT'], color='grey', alpha=0.4)
plt.scatter(ordered_df['Statutory'], my_range, color='#586AAD', alpha=1, label='Statutory CIT rate (%)')
plt.scatter(ordered_df['LACIT'], my_range, color='#AD756C', alpha=0.4 , label='LACIT rate (%)')
plt.legend(frameon=False)
sns.despine(bottom=True, left=True)
# Add title and axis names
plt.yticks(my_range, ordered_df['Country'])
plt.gca().grid(axis="x")
# Show the graph
plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/Figures/lacit_vs_statutory.svg')
plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/Figures/lacit_vs_statutory.png')


In [ ]:
##Bubble chart HS and GSW

df = pd.read_csv('C:/Users/miros/Desktop/Dropbox/Research/2005 CTHI 2021/210219_cthi_full.csv') #TODO find this file
df = df.loc[df['included_in_cthi_2021'] == 1]

countries = df['country_iso3'].tolist()
#countries = ['FRA','DEU']
for country_iso3 in countries:
    df = cthi.copy()
    df = df.loc[~(df['country_iso3'] == country_iso3)]
    x = df['cthi_2021_gsw']*100
    y = df['cthi_2021_hs']
    s = df['cthi_2021']
    plt.scatter(x, y, s, c=s, cmap="Blues", alpha=0.4, edgecolors="grey", linewidth=1)
    
    df2 = cthi.copy()
    df2 = df2.loc[df2['country_iso3'] == country_iso3]
    a = df2['cthi_2021_gsw']*100
    b = df2['cthi_2021_hs']
    c = df2['cthi_2021']
    d = df2['cthi_2021_rank']
    plt.scatter(a, b, c, color="#FEED7B", alpha=0.9, edgecolors="grey", linewidth=1)

    if c.item() > 900:
        position = "center"
        shift = 0
    else:
        position = "baseline"
        shift = 5

    # Add other elements
    plt.ylabel("Haven Score")
    plt.xlabel("Global Scale Weight (%)")
    plt.title("")
    plt.yticks(np.arange(30, 110, step=10))
    plt.axis([-1,13,20,110])
    plt.annotate(f'{country_iso3}',(a,b+shift),ha="center",va=position)

    plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Data/Code/Code_projects/202012_CTHI2021/national_press_releases/figures/{country_iso3}_scatter.png", format="png")
    plt.close()
    plt.clf()

In [ ]:
c = 5000

print(position)

In [ ]:
from matplotlib import cm
from matplotlib.ticker import LinearLocator
import numpy as np

fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Make data.
X = np.arange(0, 101, 5)
Y = np.arange(0, 14, 2)
Z = np.arange(0, 24000, 4000)
X, Y = np.meshgrid(X, Y)
R = np.cbrt(Y)
S = np.power(X,3)
Z = R*S/100

# Plot the surface.
surf = ax.plot_surface(Y, X, Z, cmap=cm.coolwarm,
                       linewidth=0, antialiased=False)

ax.zaxis.set_major_formatter('{x:,.0f}')

ax.set_xlabel('GSW (%)')
ax.set_ylabel('HS')
#ax.set_zlabel('CTHI')

# Add a color bar which maps values to colors.
fig.colorbar(surf, shrink=0.5, aspect=5)
ax.view_init(20,215)
plt.show()
plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/Figures/surface_plot.svg')

In [ ]:
from matplotlib import cm
from matplotlib.ticker import LinearLocator
import numpy as np

fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Make data.
X = np.arange(0, 101, 5)
Y = np.arange(0, 14, 2)
Z = np.arange(0, 24000, 4000)
X, Y = np.meshgrid(X, Y)
Z = X*Y

# Plot the surface.
surf = ax.plot_surface(Y, X, Z, cmap=cm.coolwarm,
                       linewidth=0, antialiased=False)

ax.zaxis.set_major_formatter('{x:,.0f}')

ax.set_xlabel('GSW (%)')
ax.set_ylabel('HS')
#ax.set_zlabel('CTHI')

# Add a color bar which maps values to colors.
fig.colorbar(surf, shrink=0.5, aspect=5)
ax.view_init(20,215)
#plt.show()
plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/Figures/surface_plot_noTransform.svg')

In [ ]:
from matplotlib import cm
from matplotlib.ticker import LinearLocator
import numpy as np

fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Make data.
X = np.arange(0, 101, 5)
Y = np.arange(0, 14, 2)
Z = np.arange(0, 24000, 4000)
X, Y = np.meshgrid(X, Y)
R = np.power(Y,0.25)
S = np.power(X,4)
Z = R*S/100

# Plot the surface.
surf = ax.plot_surface(Y, X, Z, cmap=cm.coolwarm,
                       linewidth=0, antialiased=False)

ax.zaxis.set_major_formatter('{x:,.0f}')

ax.set_xlabel('GSW (%)')
ax.set_ylabel('HS')
#ax.set_zlabel('CTHI')

# Add a color bar which maps values to colors.
fig.colorbar(surf, shrink=0.5, aspect=5)
ax.view_init(20,215)
#plt.show()
plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/Figures/surface_plot_pwr4.svg')

In [ ]:
from matplotlib import cm
from matplotlib.ticker import LinearLocator
import numpy as np

fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Make data.
X = np.arange(0, 101, 5)
Y = np.arange(0, 14, 2)
Z = np.arange(0, 24000, 4000)
X, Y = np.meshgrid(X, Y)
R = np.power(Y,0.2)
S = np.power(X,5)
Z = R*S/100

# Plot the surface.
surf = ax.plot_surface(Y, X, Z, cmap=cm.coolwarm,
                       linewidth=0, antialiased=False)

ax.zaxis.set_major_formatter('{x:,.0f}')

ax.set_xlabel('GSW (%)')
ax.set_ylabel('HS')
#ax.set_zlabel('CTHI')

# Add a color bar which maps values to colors.
fig.colorbar(surf, shrink=0.5, aspect=5)
ax.view_init(20,215)
#plt.show()
plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/Figures/surface_plot_pwr5.svg')

In [ ]:
from matplotlib import cm
from matplotlib.ticker import LinearLocator
import numpy as np

fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Make data.
X = np.arange(0, 10, 2)
Y = np.arange(0, 10, 2)
#Z = np.arange(0, 24000, 4000)
X, Y = np.meshgrid(X, Y)
Z = X*Y

# Plot the surface.
surf = ax.plot_surface(Y, X, Z, cmap=cm.coolwarm,
                       linewidth=0, antialiased=False)

ax.zaxis.set_major_formatter('{x:,.0f}')

ax.set_xlabel('GSW (%)')
ax.set_ylabel('HS')
#ax.set_zlabel('CTHI')

# Add a color bar which maps values to colors.
fig.colorbar(surf, shrink=0.5, aspect=5)
ax.view_init(20,215)
#plt.show()
plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/Figures/surface_plot_A5.svg')

In [ ]:
from matplotlib import cm
from matplotlib.ticker import LinearLocator
import numpy as np

fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Make data.
X = np.arange(0, 10, 2)
Y = np.arange(0, 10, 2)
#Z = np.arange(0, 24000, 4000)
X, Y = np.meshgrid(X, Y)
Z = (X+Y)/2

# Plot the surface.
surf = ax.plot_surface(Y, X, Z, cmap=cm.coolwarm,
                       linewidth=0, antialiased=False)

ax.zaxis.set_major_formatter('{x:,.0f}')

ax.set_xlabel('GSW (%)')
ax.set_ylabel('HS')
#ax.set_zlabel('CTHI')

# Add a color bar which maps values to colors.
fig.colorbar(surf, shrink=0.5, aspect=5)
ax.view_init(20,215)
#plt.show()
plt.savefig(f'{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/Results/Figures/surface_plot_A5.svg')